# Lab 3 - Search on Unstructured Data: from Ctrl-F to an Enterprise Search Service

**Day 1 - AI Solution Architect programme | ~60-75 min | demo-led, you follow along**

---

### The scenario

> *Northwind Advisory is a 4,000-person professional-services firm. Its knowledge lives in 48 (here; 480,000 in real life) messy documents:
> policies, how-tos, post-mortems, lessons learned, tickets, status e-mails. Every week the helpdesk answers the same questions
> ("How many weeks of parental leave?", "VPN says ERR-4021!") because **nobody can find the answer**.*
> The COO has asked you - the solution architect - for a search service. Not a demo that works on three hand-picked queries; one you can **defend in front of the CISO and the CFO**.

### What you will be able to do after this lab

1. Explain *why* keyword search fails on natural language, and build the keyword baselines (inverted index, TF-IDF, BM25) from scratch.
2. Treat **evaluation as a first-class citizen**: write Recall@k, MRR, nDCG@k and score every design decision against a gold set.
3. Explain embeddings, cosine similarity, and what an embedding model is (and is **not**).
4. Choose a **chunking** strategy and justify it with numbers.
5. Implement exact vector search, understand why *approximate* nearest-neighbour (ANN) indexes exist, and choose a vector store.
6. Apply **metadata filters, freshness rules and security trimming** (before ranking, not after!).
7. Combine lexical and semantic signals with **hybrid search** (RRF), and decide when the system should say *"I don't know"*.
8. Assemble a grounded, cited prompt - the bridge to **RAG** on Day 3 - and draw the production architecture.

### How the notebook works

* Cells marked **Exercise** have a stub with a `TODO` and a hint. Try for 3-5 minutes; the instructor will walk through a solution.
* Everything is deterministic (seeds fixed). Runs top-to-bottom in a few minutes on a laptop; no GPU, no API key required.
* Our mantra: **measure before you optimise.** We build the evaluation harness *before* we build the clever stuff.

In [ ]:
# ---- 0. Environment check & shared setup -------------------------------------------------
import os, re, json, math, time, hashlib, textwrap, warnings, platform, sys
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, scipy

warnings.filterwarnings("ignore")
SEED = 42                       # one seed to rule them all
rng = np.random.default_rng(SEED)
np.random.seed(SEED)

# Relative paths only: the notebook runs with cwd = the labs directory.
DATA   = Path("data")
CORPUS = DATA / "corpus"
ART    = Path("artifacts"); ART.mkdir(exist_ok=True)      # caches and figures live here
assert CORPUS.exists(), "Run this notebook from the folder that contains data/ (data/corpus not found)."

# Workshop colour palette (used in every chart)
INK, TEAL, LIME, CORAL, GOLD = "#0B1F2A", "#0E7C86", "#9BD13B", "#E4572E", "#F2B134"
plt.rcParams.update({
    "figure.dpi": 110, "axes.edgecolor": INK, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": INK, "ytick.color": INK, "axes.spines.top": False, "axes.spines.right": False,
    "axes.prop_cycle": plt.cycler(color=[TEAL, CORAL, GOLD, LIME, INK]), "font.size": 10,
})
pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 70)
pd.set_option("display.float_format", lambda v: f"{v:.3f}")

print(f"python {platform.python_version()} | numpy {np.__version__} | pandas {pd.__version__} | "
      f"sklearn {sklearn.__version__} | scipy {scipy.__version__}")
try:
    import fastembed
    print("fastembed available:", getattr(fastembed, "__version__", "yes"))
except Exception as e:                       # the lab still works without it (LSA fallback later)
    print("fastembed NOT available -> the lab will use the TF-IDF+SVD fallback.", e)
print("Corpus files found:", len(list(CORPUS.glob("*.txt"))))

## 1. The corpus - and the pain of Ctrl-F

**Architect's lens:** before choosing any technology, *look at your data*. Search quality is bounded by what is in the documents:
their length, their mess, their metadata. Our corpus is deliberately realistic: headers, bullet lists, abbreviations, typos, signature blocks,
two *versions* of some policies, and documents of very different lengths.

Each file starts with a small metadata header (`title`, `source`, `department`, `date`, `sensitivity`, sometimes `policy_id`), a blank line, then the body.
That header is gold: later it powers filters, freshness and access control. In real life it comes from the connector (SharePoint, Confluence, ServiceNow...).

In [ ]:
def parse_doc(path: Path) -> dict:
    """Split 'key: value' header lines from the free-text body (first blank line is the separator)."""
    raw = path.read_text(encoding="utf-8")
    head, _, body = raw.partition("\n\n")
    meta = {}
    for line in head.splitlines():
        k, _, v = line.partition(":")
        meta[k.strip()] = v.strip()
    return {"doc_id": path.name.split("_")[0], "filename": path.name, **meta, "body": body.strip()}

docs = pd.DataFrame([parse_doc(p) for p in sorted(CORPUS.glob("*.txt"))])
docs["date"] = pd.to_datetime(docs["date"])
docs["n_words"] = docs["body"].str.split().str.len()
docs = docs.set_index("doc_id", drop=False)
DOC_TEXT = {d: f"{r.title}\n{r.body}" for d, r in docs.iterrows()}      # what we index at document level

print(f"{len(docs)} documents | {docs.n_words.sum():,} words | dates {docs.date.min().date()} -> {docs.date.max().date()}")
display(docs[["title", "department", "date", "sensitivity", "n_words"]].head(8))

In [ ]:
# ---- Quick EDA: length distribution, departments, sensitivity --------------------------------
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
ax[0].hist(docs.n_words, bins=14, color=TEAL, edgecolor="white")
ax[0].axvline(docs.n_words.median(), color=CORAL, ls="--", label=f"median = {docs.n_words.median():.0f} words")
ax[0].set(title="Document length (words)", xlabel="words", ylabel="documents"); ax[0].legend(frameon=False)

dept = docs.department.value_counts().sort_values()
ax[1].barh(dept.index, dept.values, color=TEAL); ax[1].set_title("Documents per department")

sens = docs.sensitivity.value_counts().reindex(["public", "internal", "confidential"])
ax[2].bar(sens.index, sens.values, color=[LIME, GOLD, CORAL]); ax[2].set_title("Sensitivity label")
plt.tight_layout(); plt.savefig(ART / "eda.png"); plt.show()

print("Longest documents (these will need chunking):")
display(docs.nlargest(5, "n_words")[["title", "n_words"]])
print("Versioned policies (same policy_id, different dates):")
dup = docs[docs.policy_id.notna() & docs.duplicated("policy_id", keep=False)]
display(dup.sort_values(["policy_id", "date"])[["policy_id", "title", "date"]])

### The pain of Ctrl-F

Exact substring matching is what most people (and, honestly, many intranet search boxes) do. Let's see how a human *thinks* versus what a document *says*.
Watch the **zero-hit** rows - each one is a frustrated employee who then pings the helpdesk.

In [ ]:
def ctrl_f(term: str) -> list[str]:
    """Case-insensitive substring search - the 'find in page' of the whole corpus."""
    t = term.lower()
    return [d for d, text in DOC_TEXT.items() if t in text.lower()]

human_phrases = ["Spain", "month", "work from home", "remote work", "reimbursement", "refund",
                 "expense claim", "laptop broke", "device replacement", "ERR-4021", "err 4021"]
rows = [{"what the human typed": p, "docs hit": len(ctrl_f(p)), "which": ", ".join(ctrl_f(p)[:6])} for p in human_phrases]
display(pd.DataFrame(rows))
print("\nNotice: 'Spain' -> 0 hits, yet doc D05 answers 'Can I work from Spain for a month?'. The words simply never appear.")
print("Also 'ERR-4021' works but 'err 4021' (what people actually type) does not.")

#### Exercise 1 - How many ways can you say "paternity leave"?  (3 min)

Using `ctrl_f`, test five phrases an employee might type when looking for the *new-parent leave policy*
(think: `maternity`, `paternity`, `new baby`, `time off`, `parental`...). Which ones find D01?
The lesson: **vocabulary mismatch** - the #1 reason keyword search disappoints.

In [ ]:
# TODO: fill the list with 5 phrases and print, for each, whether D01 is among the hits.
# Hint: reuse ctrl_f(); `"D01" in ctrl_f(p)` gives a boolean.
my_phrases = []          # <- your phrases here
for p in my_phrases:
    pass                 # <- print phrase and whether D01 was found

## 2. Level 1 - Keyword search: inverted index, TF-IDF, BM25

### Why an inverted index?
A book has an **index at the back**: *word -> pages*. A search engine does the same: `term -> list of (document, count)` (a *postings list*).
To answer a query you don't scan every document, you look up each query term and combine the lists. This is how Lucene, Elasticsearch, OpenSearch,
and Azure AI Search keyword search work under the hood - at huge scale, with compression - but the idea fits in 15 lines.

Pre-processing decides what counts as "the same word":
* **tokenise** (lower-case, split; we keep identifiers like `ERR-4021` and `POL-HR-017` as single tokens *and* their parts),
* drop **stop-words** (`the`, `is`...) that carry no signal,
* **light stemming** so `claims`/`claimed` -> `claim`. (Stemming is a blunt tool: it will *not* turn `stole` into `stolen`.)

In [ ]:
STOP = set("""a an the and or of to in on for with at by from is are was were be been it this that these those as if you your i we our
can do does how what who when where which my me will would should may must also any all per than then so there their they them his her its
into over under about up out more most such only own same too very s t am has have had get got""".split())
TOKEN_RE = re.compile(r"[a-z0-9]+(?:[-_][a-z0-9]+)+|[a-z0-9]+")     # keeps ERR-4021, POL-HR-017 whole

def light_stem(w: str) -> str:
    """Tiny suffix stripper: claims->claim, policies->policy. (Porter/Snowball do this more carefully.)"""
    if not w.isalpha():
        return w
    if len(w) > 5 and w.endswith("ies"):
        return w[:-3] + "y"
    for suf in ("ing", "ed", "es", "s"):
        if len(w) > len(suf) + 3 and w.endswith(suf):
            return w[: -len(suf)]
    return w

def tokenize(text: str) -> list[str]:
    toks = []
    for m in TOKEN_RE.findall(text.lower()):
        pieces = [m] + (re.split(r"[-_]", m) if re.search(r"[-_]", m) else [])   # 'err-4021' -> err-4021, err, 4021
        for p in pieces:
            if p in STOP or len(p) < 2:
                continue
            toks.append(light_stem(p))
    return toks

print(tokenize("Error ERR-4021: the VPN gateway authentication failed after my password changes!"))

In [ ]:
def build_inverted_index(texts: dict[str, str]):
    """term -> {doc_id: term_frequency}; also returns document lengths (in tokens)."""
    index, lengths = defaultdict(dict), {}
    for doc_id, text in texts.items():
        toks = tokenize(text)
        lengths[doc_id] = len(toks)
        for term, tf in Counter(toks).items():
            index[term][doc_id] = tf
    return index, lengths

INDEX, DOC_LEN = build_inverted_index(DOC_TEXT)
print(f"vocabulary: {len(INDEX):,} terms | avg doc length: {np.mean(list(DOC_LEN.values())):.0f} tokens")
for term in ["vpn", "err-4021", "reimbursement", "spain"]:
    print(f"  postings[{term!r}] = {INDEX.get(term, {})}")
df_count = pd.Series({t: len(p) for t, p in INDEX.items()}).sort_values(ascending=False)
print("\nMost common terms (document frequency) - these carry the *least* information:", list(df_count.head(8).index))

### Boolean retrieval = set algebra on postings
`AND` = intersection, `OR` = union, `NOT` = difference. Boolean search is exact and explainable (auditors love it) but gives no *ranking* and no tolerance for vocabulary mismatch.

#### Exercise 2 - Boolean query builder (5 min)
Write `boolean_query(index, must=[], any_of=[], must_not=[])` returning the set of doc ids that contain **all** `must` terms, **at least one** of `any_of`, and **none** of `must_not`.
Then answer: *which documents mention a password **or** MFA, but are not about the VPN?*

In [ ]:
def boolean_query(index, must=(), any_of=(), must_not=()):
    """TODO: return a set of doc_ids.
    Hint: normalise each term with tokenize(term)[0] so it matches the index; start from the set of ALL doc ids;
    intersect for `must`, intersect with the union for `any_of` (only if provided), subtract for `must_not`."""
    return set()

### Ranking: TF-IDF and BM25
Boolean says *which* documents match; we want *which match best*. Two intuitions:

* **TF (term frequency):** a document that mentions "VPN" 6 times is more about VPN than one mentioning it once - but with diminishing returns (the 20th mention adds little).
* **IDF (inverse document frequency):** a term in 90% of documents (`policy`) tells you nothing; a term in 2 documents (`err-4021`) is a strong clue. `idf = log(N / df)`.

**TF-IDF + cosine** multiplies these and compares query and document as vectors. **BM25** (the default in Lucene/Elasticsearch/OpenSearch for 15+ years) is a better-behaved relative:

$$\text{score}(q,d)=\sum_{t\in q}\ \underbrace{\ln\!\Big(1+\tfrac{N-df_t+0.5}{df_t+0.5}\Big)}_{\text{IDF: rare terms count more}}\cdot\underbrace{\frac{tf_{t,d}\,(k_1+1)}{tf_{t,d}+k_1\big(1-b+b\,\frac{|d|}{avgdl}\big)}}_{\text{saturating TF with length normalisation}}$$

* `k1` (~1.2-2.0) controls TF saturation: how quickly repeating a word stops helping.
* `b` (0..1) controls length normalisation: long documents naturally contain more words, so we discount them. `b=0` ignores length; `b=1` normalises fully.

We implement it ourselves - it's short, and then Elasticsearch stops being magic. First sklearn's TF-IDF as the quick baseline.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

DOC_IDS = list(docs.doc_id)
tfidf = TfidfVectorizer(analyzer=tokenize, sublinear_tf=True)            # our tokenizer, sklearn's maths
X_tfidf = tfidf.fit_transform([DOC_TEXT[d] for d in DOC_IDS])             # rows are L2-normalised

def rank_scores(scores, ids, k=10, drop_zero=True):
    """Turn a score array into a ranked list [(id, score), ...]. Keyword engines return nothing for score 0."""
    order = np.argsort(-scores, kind="stable")[:k]
    return [(ids[i], float(scores[i])) for i in order if (scores[i] > 0 or not drop_zero)]

def tfidf_search(query, k=10):
    sims = linear_kernel(tfidf.transform([query]), X_tfidf).ravel()      # cosine (rows already normalised)
    return rank_scores(sims, DOC_IDS, k)

class BM25:
    """Okapi BM25 from scratch. fit() builds postings; scores() returns one score per document."""
    def __init__(self, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
    def fit(self, texts):
        self.N = len(texts)
        toks = [tokenize(t) for t in texts]
        self.dl = np.array([len(t) for t in toks], dtype=float)
        self.avgdl = self.dl.mean()
        self.postings = defaultdict(list)                                 # term -> [(doc_idx, tf)]
        for i, t in enumerate(toks):
            for term, tf in Counter(t).items():
                self.postings[term].append((i, tf))
        self.idf = {t: math.log(1 + (self.N - len(p) + 0.5) / (len(p) + 0.5)) for t, p in self.postings.items()}
        return self
    def scores(self, query):
        s = np.zeros(self.N)
        for term in set(tokenize(query)):                                 # query-term frequency is ignored
            for i, tf in self.postings.get(term, []):
                norm = tf + self.k1 * (1 - self.b + self.b * self.dl[i] / self.avgdl)
                s[i] += self.idf[term] * tf * (self.k1 + 1) / norm
        return s

bm25_docs = BM25().fit([DOC_TEXT[d] for d in DOC_IDS])
def bm25_search(query, k=10):
    return rank_scores(bm25_docs.scores(query), DOC_IDS, k)

for q in ["ERR-4021", "how do I get my money back for a client dinner"]:
    print(f"\nQuery: {q!r}")
    for name, fn in [("TF-IDF", tfidf_search), ("BM25  ", bm25_search)]:
        print(f"  {name}: ", [(d, round(s, 2)) for d, s in fn(q, 3)])

### Where does keyword search break?
For each *hand-picked semantic trap* below we ask: **at which rank does the right document appear?** Rank 1 is perfect; `-` means "not retrieved at all".
(We will turn this into proper metrics in a moment - but looking at raw rankings builds intuition that a single number never will.)

In [ ]:
GOLD_SHOWCASE = {                     # query -> the document a human would say is *the* answer
    "can I work from Spain for a month": "D05",
    "my laptop broke, what now": "D25",
    "someone stole my phone, what should I do": "D20",
    "I'm feeling burnt out, who can I talk to": "D09",
    "ERR-4021": "D26",
    "POL-SEC-012": "D22",
}
def rank_of(search_fn, query, gold, k=48):
    ranked = [d for d, _ in search_fn(query, k)]
    return ranked.index(gold) + 1 if gold in ranked else None

fmt = lambda r: "-" if r is None else str(r)                      # '-' = not retrieved at all
rows = [{"query": q, "gold": g, "TF-IDF rank": fmt(rank_of(tfidf_search, q, g)), "BM25 rank": fmt(rank_of(bm25_search, q, g))}
        for q, g in GOLD_SHOWCASE.items()]
display(pd.DataFrame(rows))
print("Semantic questions: the gold document is missing or buried. Identifier lookups: keyword search is perfect.")
print("-> We need BOTH worlds eventually (hybrid). But first: how do we *measure*?")

## 3. The evaluation harness - build it *first*

> *"If you can't measure it, you can't improve it - and you can't defend it in a steering committee."*

A search demo that "looks good on five queries" is an anecdote. A **golden set** (query -> which documents are relevant) turns arguments into numbers.
Ours (`data/eval_queries.json`) has four kinds of query, designed to probe different failure modes:

| type | what it tests | example |
|---|---|---|
| `semantic` | paraphrase / synonyms - little word overlap | *"can I work from Spain for a month"* |
| `keyword` | exact identifiers - error codes, policy IDs, product names | *"ERR-4021"* |
| `hybrid` | an exact token **plus** meaning | *"VPN keeps failing with ERR-4021 after I changed my password"* |
| `no_answer` | the corpus has **no** answer - the system should abstain | *"what is the dress code for client meetings"* |

**Three metrics, each answering a different business question** (binary relevance, top-`k` list):

* **Recall@k** - *"Did the right document make the first page at all?"* (fraction of relevant docs found in the top k). Critical for RAG: if the answer isn't retrieved, the LLM can't use it.
* **MRR** (mean reciprocal rank) - *"How far down must the user scroll?"* `1/rank` of the first relevant result. Great for single-answer lookups.
* **nDCG@k** - *"Is the ordering good?"* Rewards relevant results near the top with a logarithmic discount, normalised so 1.0 is a perfect list.

Because this is *so* small, we write the metrics ourselves rather than import them - and so you know exactly what the numbers mean.

In [ ]:
EVAL = json.loads((DATA / "eval_queries.json").read_text())
eval_df = pd.DataFrame(EVAL)
print(f"{len(EVAL)} gold queries:", eval_df.type.value_counts().to_dict())
display(eval_df.sample(5, random_state=SEED)[["query", "relevant_doc_ids", "type"]])

# ---- the three metrics, hand-written ------------------------------------------------------
def recall_at_k(ranked, relevant, k):
    """Fraction of the relevant docs that appear in the top k."""
    return len(set(ranked[:k]) & set(relevant)) / len(relevant)

def reciprocal_rank(ranked, relevant):
    """1/rank of the first relevant hit (0 if none)."""
    for i, d in enumerate(ranked, start=1):
        if d in relevant:
            return 1.0 / i
    return 0.0

def ndcg_at_k(ranked, relevant, k):
    """Binary-relevance nDCG: DCG of our list / DCG of the ideal list. Discount = 1/log2(rank+1)."""
    dcg = sum(1.0 / math.log2(i + 1) for i, d in enumerate(ranked[:k], start=1) if d in relevant)
    ideal = sum(1.0 / math.log2(i + 1) for i in range(1, min(len(relevant), k) + 1))
    return dcg / ideal

# sanity checks on tiny hand-made cases - always unit-test your metrics!
assert recall_at_k(["a", "b", "c"], ["b", "z"], 3) == 0.5
assert reciprocal_rank(["a", "b", "c"], ["c"]) == 1 / 3
assert abs(ndcg_at_k(["a", "b"], ["a"], 2) - 1.0) < 1e-9
assert ndcg_at_k(["x", "a"], ["a"], 2) < 1.0
print("metric unit tests passed")

#### Exercise 3 - Add a metric (4 min)
Add **Precision@k** (what fraction of the top k results are relevant?). Think about when precision matters more than recall:
an **LLM context window** that you do not want to fill with noise - or a user who only reads the first three hits.

In [ ]:
def precision_at_k(ranked, relevant, k):
    """TODO: fraction of the top-k results that are relevant.
    Hint: len(set(ranked[:k]) & set(relevant)) / k"""
    return 0.0

### `evaluate(search_fn)` - one function to score every system

The contract is deliberately tiny so that **every** system we build today plugs in: `search_fn(query) -> [(doc_id, score), ...]` ranked best-first.
`evaluate` returns a table broken down **by query type** - averages hide the story. (A system that is superb at IDs and hopeless at paraphrase can still post a decent average.)
`no_answer` queries have no relevant documents, so ranking metrics don't apply; we record the top-1 score instead, which we will use later to calibrate an "I don't know" threshold.

In [ ]:
K = 5

def evaluate(search_fn, k=K, name="system"):
    """Score a search function on the gold set. Returns a per-type summary DataFrame;
    per-query details are attached as summary.attrs['per_query']."""
    rows = []
    for e in EVAL:
        ranked, top1 = [], 0.0
        res = search_fn(e["query"])
        seen = set()
        for d, s in res:                     # de-duplicate: document-level evaluation
            if d not in seen:
                ranked.append(d); seen.add(d)
        top1 = res[0][1] if res else 0.0
        rel = e["relevant_doc_ids"]
        row = {"query": e["query"], "type": e["type"], "top1_score": top1, "first_hit": None}
        if rel:
            row.update({f"Recall@{k}": recall_at_k(ranked, rel, k), "MRR": reciprocal_rank(ranked, rel),
                        f"nDCG@{k}": ndcg_at_k(ranked, rel, k)})
            row["first_hit"] = next((i for i, d in enumerate(ranked, 1) if d in rel), None)
        rows.append(row)
    per_q = pd.DataFrame(rows)
    metrics = [f"Recall@{k}", "MRR", f"nDCG@{k}"]
    ans = per_q[per_q.type != "no_answer"]
    out = [{name: typ, "n": len(g), **g[metrics].mean().to_dict()} for typ, g in ans.groupby("type")]
    out.append({name: "ALL (answerable)", "n": len(ans), **ans[metrics].mean().to_dict()})
    by_type = pd.DataFrame(out).set_index(name)
    by_type.attrs["per_query"] = per_q
    return by_type

LEADERBOARD = {}                                   # name -> summary table; filled as we go
def score_system(name, search_fn, show=True):
    res = evaluate(search_fn, name=name)
    LEADERBOARD[name] = res
    if show:
        display(res)
    return res

score_system("BM25 (doc level)", bm25_search)
score_system("TF-IDF (doc level)", tfidf_search);

**Read the table like an architect.** Keyword engines are strong on `keyword` queries (that is their home turf) and collapse on `semantic` ones.
No amount of BM25 parameter-tuning will fix *vocabulary mismatch* - it's a representation problem, not a tuning problem. That motivates the next idea: **embeddings**.

## 4. Embeddings: representing *meaning* as geometry

### What is an embedding model?
A neural network trained on huge amounts of text to map a piece of text to a point (a vector of, here, 384 numbers) such that **texts with similar meaning land near each other**.
Think of a library where books are shelved by *topic similarity* rather than alphabetically - and the "shelf coordinates" have 384 dimensions.

**What it is:** a frozen function `text -> vector`, learned from data. Similar meaning -> small angle between vectors.
**What it is not:**
* not a database, not a search engine (it's only the *representation*; search is a separate step);
* not "understanding": it captures topical/semantic similarity, **not truth** - "we can claim alcohol" and "we cannot claim alcohol" land very close (you'll prove this in Exercise 4);
* not domain-proof: trained on general web text, it may blur your jargon (`PE` = permanent establishment? physical education?) and **identifiers** (`ERR-4021` vs `ERR-4107`);
* not language-agnostic unless it's a *multilingual* model (this one, `bge-small-en`, is English-only; for a European firm you'd evaluate `multilingual-e5` or `bge-m3` class models);
* **a versioned dependency**: change the model and every stored vector becomes meaningless - you must re-embed *everything*. Treat `model name + version + dimension` like a database schema.

We use `BAAI/bge-small-en-v1.5` via `fastembed` (ONNX runtime, runs on CPU, ~130 MB, 384 dimensions). If the model can't be downloaded (e.g. offline classroom Wi-Fi),
the next cell **falls back** to TF-IDF + SVD ("LSA") vectors - a classic, weaker way to get dense semantic-ish vectors - so the rest of the lab still runs. Watch for the warning.

In [ ]:
from sklearn.decomposition import TruncatedSVD, PCA

class Embedder:
    """text -> L2-normalised float32 vectors. Tries a real neural model; falls back to LSA (TF-IDF + SVD)."""
    QUERY_PREFIX = "Represent this sentence for searching relevant passages: "     # BGE's recommended query instruction

    def __init__(self, model_name="BAAI/bge-small-en-v1.5"):
        self.model_name, self.model = model_name, None
        try:
            from fastembed import TextEmbedding
            self.model = TextEmbedding(model_name)
            self.dim = len(next(iter(self.model.embed(["warm-up"]))))
            self.kind = model_name
        except Exception as e:
            print("=" * 100)
            print(f"WARNING: could not load '{model_name}' ({type(e).__name__}: {str(e)[:80]}).")
            print("         Falling back to TF-IDF + TruncatedSVD (LSA) 'embeddings'. The lab still runs,")
            print("         but semantic queries will score LOWER than with a neural model. Re-run when online.")
            print("=" * 100)
            self.kind, self.dim = "lsa-fallback", 96
            self._lsa = None

    def fit(self, texts):
        """Only the LSA fallback needs fitting (on the corpus); neural models are pre-trained."""
        if self.model is None:
            vec = TfidfVectorizer(analyzer=tokenize, sublinear_tf=True)
            X = vec.fit_transform(texts)
            n_comp = min(self.dim, X.shape[0] - 1, X.shape[1] - 1)
            self._lsa = (vec, TruncatedSVD(n_components=n_comp, random_state=SEED).fit(X))
        return self

    def _raw(self, texts, is_query):
        if self.model is not None:
            texts = [self.QUERY_PREFIX + t for t in texts] if is_query else list(texts)
            return np.array(list(self.model.embed(texts, batch_size=32)), dtype=np.float32)
        vec, svd = self._lsa
        return svd.transform(vec.transform(texts)).astype(np.float32)

    def encode(self, texts, is_query=False, cache=False):
        """Return unit-length vectors (so dot product == cosine). Corpus embeddings are cached on disk."""
        texts = list(texts)
        path = None
        if cache:
            key = hashlib.sha1(("|".join([self.kind, str(is_query)] + texts)).encode()).hexdigest()[:16]
            (ART / "emb_cache").mkdir(exist_ok=True)
            path = ART / "emb_cache" / f"{key}.npy"
            if path.exists():
                return np.load(path)
        V = self._raw(texts, is_query)
        V /= np.linalg.norm(V, axis=1, keepdims=True) + 1e-12            # L2-normalise
        if path is not None:
            np.save(path, V)
        return V

t0 = time.time()
EMB = Embedder()
EMB.fit(list(DOC_TEXT.values()))
print(f"embedder: {EMB.kind} | dim = {EMB.dim} | ready in {time.time() - t0:.1f}s")

### Cosine similarity, dot product, L2 distance - which one?
For two vectors `a`, `b`: **cosine** = angle between them (ignores length); **dot** = cosine x lengths; **L2** = straight-line distance.
If every vector is **normalised to length 1** (we do this), all three give *the same ranking*: `L2^2 = 2 - 2*cos` and `dot = cos`. That's why most vector stores
ask you to pick a metric at index-creation time - and why normalising up-front lets you use the cheapest one (dot product = one matrix multiply).
Mismatching the metric the model was trained for is a classic silent bug.

Next, a feel for the geometry: eight hand-picked sentences, two per topic, plus an outsider. We expect a **block structure** on the diagonal.

In [ ]:
sentences = [
    "My laptop is broken",                          # IT hardware
    "I need a replacement computer",
    "How do I claim back my taxi fare?",            # expenses
    "Submitting an expense reimbursement",
    "How many days of annual leave do I get?",      # leave
    "Time off for new parents",
    "Reset my multi-factor authentication",         # security
    "The cat sat on the sofa",                      # outsider
]
S = EMB.encode(sentences)
sim = S @ S.T                                       # cosine similarity matrix (vectors are unit length)

fig, ax = plt.subplots(figsize=(7.2, 6))
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list("nw", ["#FFFFFF", TEAL, INK])
im = ax.imshow(sim, cmap=cmap, vmin=min(0.0, sim.min()), vmax=1)
ax.set_xticks(range(len(sentences))); ax.set_yticks(range(len(sentences)))
ax.set_xticklabels([s[:22] for s in sentences], rotation=40, ha="right"); ax.set_yticklabels([s[:30] for s in sentences])
for i in range(len(sentences)):
    for j in range(len(sentences)):
        ax.text(j, i, f"{sim[i, j]:.2f}", ha="center", va="center", fontsize=8, color="white" if sim[i, j] > 0.55 else INK)
ax.set_title(f"Cosine similarity ({EMB.kind})"); plt.colorbar(im, shrink=0.8); plt.tight_layout()
plt.savefig(ART / "sim_heatmap.png"); plt.show()

a, b = S[0], S[1]
print("cosine  :", round(float(a @ b), 4))
print("dot     :", round(float(a @ b), 4), "  (identical, vectors are unit length)")
print("L2^2    :", round(float(((a - b) ** 2).sum()), 4), " = 2 - 2*cos =", round(2 - 2 * float(a @ b), 4))
print("scaling a by 3 changes dot to", round(float((3 * a) @ b), 3), "but cosine stays", round(float((3 * a) @ b / (np.linalg.norm(3 * a) * np.linalg.norm(b))), 4))

#### Exercise 4 - Fool the embedding model (4 min)
Embeddings capture *topic*, not *truth*. Write **three sentence pairs** that mean the **opposite** (or differ in one critical detail - a number, a negation, an entity)
and see how similar the model thinks they are. Compare with an unrelated pair. Architect take-away: embeddings alone must never be the last line of defence
for facts like "26 weeks vs 16 weeks" - that is what **citations, filters and evaluation** are for.

In [ ]:
# TODO: add two more pairs (negation, different number, different region...) and print the cosine for each.
pairs = [
    ("Employees can claim alcohol on expenses", "Employees cannot claim alcohol on expenses"),
    # ("...", "..."),
]
for a_txt, b_txt in pairs:
    pass          # <- encode both with EMB.encode([...]) and print their cosine

### Seeing the whole corpus: a 2-D map
We embed every **document** (title + body) and squash 384 dimensions into 2 with **PCA** (find the two directions of greatest variance). It's a lossy picture
- distances are only roughly preserved - but if the map shows departments clustering, the embedding space is encoding something real about our domain.

*One architect's warning hidden here:* embedding models have a **maximum input length** (512 tokens for BGE-small; anything longer is silently truncated!). We'll flag which documents suffer.

In [ ]:
DOC_EMB = EMB.encode([DOC_TEXT[d] for d in DOC_IDS], cache=True)
print("doc embedding matrix:", DOC_EMB.shape, "| row norms ~", np.round(np.linalg.norm(DOC_EMB, axis=1)[:3], 3))

est_tokens = docs.n_words * 1.35                           # rough: English ~1.3-1.4 tokens per word
print(f"Documents likely to exceed a 512-token window (silent truncation): {list(docs.doc_id[est_tokens > 480])}")

pca = PCA(n_components=2, random_state=SEED)
XY = pca.fit_transform(DOC_EMB)
dept_colors = dict(zip(sorted(docs.department.unique()), [TEAL, CORAL, GOLD, LIME, INK, "#7A9E9F", "#B5651D", "#6C4F9E"]))
fig, ax = plt.subplots(figsize=(9.5, 6.4))
for dep, col in dept_colors.items():
    m = (docs.department == dep).values
    ax.scatter(XY[m, 0], XY[m, 1], s=70, color=col, edgecolor="white", label=dep, zorder=3)
for i, d in enumerate(DOC_IDS):
    ax.annotate(d, XY[i], fontsize=7, xytext=(3, 3), textcoords="offset points", color=INK)
ax.set(title=f"Documents in embedding space (PCA, {pca.explained_variance_ratio_.sum():.0%} of variance kept)",
       xlabel="PC1", ylabel="PC2")
ax.legend(frameon=False, ncol=2, fontsize=8); plt.tight_layout(); plt.savefig(ART / "pca_docs.png"); plt.show()

### Nearest neighbours: does "close" mean "relevant"?
For a few documents we list their closest neighbours. Look for two things: (1) sensible topical neighbours, and (2) **near-duplicates** (the 2022 vs 2024 versions of the same policy) -
which are *so* close that the vector index alone can never tell you which one is current. That's a metadata job (section 7).

In [ ]:
def neighbours(doc_id, k=3):
    i = DOC_IDS.index(doc_id)
    sims = DOC_EMB @ DOC_EMB[i]
    order = [j for j in np.argsort(-sims) if j != i][:k]
    return [(DOC_IDS[j], round(float(sims[j]), 3), docs.title.iloc[j][:55]) for j in order]

for d in ["D01", "D13", "D26", "D46"]:
    print(f"\n{d}: {docs.title[d][:70]}")
    for nid, s, t in neighbours(d):
        print(f"    {s:.3f}  {nid}  {t}")

print("\n(The classic 'king - man + woman = queen' analogy is a word2vec curiosity; sentence-embedding models like this one don't support such")
print(" arithmetic reliably, so we skip it - similarity search is the property we can actually rely on in production.)")

### The payoff: same question, two retrieval engines
Now the moment we've been building toward. We embed the *query* (note: queries and documents are embedded by the same model, and BGE adds a short instruction prefix to queries) and rank documents by cosine similarity.
Compare with BM25 on the semantic-trap queries.

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=None)
def embed_query(q: str) -> np.ndarray:
    return EMB.encode([q], is_query=True)[0]

def dense_doc_search(query, k=10):
    sims = DOC_EMB @ embed_query(query)
    return rank_scores(sims, DOC_IDS, k, drop_zero=False)

for q in ["can I work from Spain for a month", "someone stole my phone, what should I do", "my laptop broke, what now"]:
    print(f"\nQUERY: {q!r}")
    print("  BM25  :", [(d, round(s, 2)) for d, s in bm25_search(q, 3)] or "no results")
    print("  Dense :", [(d, round(s, 3)) for d, s in dense_doc_search(q, 3)])

score_system("Dense (doc level)", dense_doc_search)
print("\nCaveat: at document level, long documents get truncated and diluted into one vector - chunking (next) should help.")

## 5. Chunking: what exactly do we put in the index?

Notice the weakness we just saw: one vector per **document** forces a 400-word policy about *ten* topics into a single point. The query "someone stole my phone" is answered by *one paragraph* of the 370-word InfoSec policy.
Two problems follow:

1. **Dilution / truncation:** the embedding of a long document is a blurry average (and anything past the model's input window is ignored).
2. **Context quality:** whatever we retrieve is what we feed a human - or later an LLM - to read. Retrieving the whole policy to find one sentence wastes tokens (cost, latency) and hides the answer ("lost in the middle").

So we split documents into **chunks**, embed each chunk, retrieve chunks, and roll results back up to documents. The art is choosing the boundaries:

| strategy | idea | pros | cons |
|---|---|---|---|
| **fixed-size + overlap** | every N words, overlapping by ~20% | trivial, predictable cost | cuts mid-sentence / mid-table; overlap duplicates text |
| **paragraph** | split at blank lines, merge to a target size | respects author's structure | paragraphs vary wildly in size |
| **heading-aware** | split at headings/numbered sections, prepend `title > heading` to each chunk | chunks are self-contained and *labelled*; best citations | needs parsing; messy docs have no headings |
| **sentence-window** | embed single sentences, return +/- neighbours | precise matching, rich context | many vectors; more storage |

**Provenance rule (architects, write this on your hand):** every chunk carries `doc_id`, `chunk_id`, and the document's metadata (`department`, `date`, `sensitivity`, `policy_id`...).
Without it you can't cite, can't filter, can't apply access control and can't delete a document's chunks when it's retired.

In [ ]:
# ---- Three chunkers. Each takes a document body and returns [(section_heading, chunk_text), ...] -----
def fixed_chunks(text, size=120, overlap=0.2):
    """Slide a window of `size` words, stepping by size*(1-overlap). Keeps the original text slice."""
    words = list(re.finditer(r"\S+", text))
    step = max(1, int(size * (1 - overlap)))
    out = []
    for start in range(0, len(words), step):
        seg = words[start:start + size]
        out.append(("", text[seg[0].start():seg[-1].end()]))
        if start + size >= len(words):
            break
    return out

def paragraph_chunks(text, target=120):
    """Split at blank lines; greedily merge paragraphs until ~target words."""
    paras = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    out, cur, n = [], [], 0
    for p in paras:
        w = len(p.split())
        if cur and n + w > target:
            out.append(("", "\n\n".join(cur))); cur, n = [], 0
        cur.append(p); n += w
    if cur:
        out.append(("", "\n\n".join(cur)))
    return out

def is_heading(line: str) -> bool:
    """Heuristics for messy corporate text: 'WHAT WENT WELL', '3. Entitlement', 'Q: When is the review cycle?'"""
    words = line.split()
    if re.match(r"^Q[.:]\s", line):                                                       # FAQ question
        return True
    if re.match(r"^\d{1,2}\.\s+\S", line) and len(words) <= 8 and not line.endswith((".", ":")):   # numbered, short
        return True
    first = re.sub(r"[^A-Za-z]", "", words[0]) if words else ""
    return len(words) <= 12 and len(first) >= 3 and words[0].replace("-", "").isalpha() and words[0].isupper() and not line.endswith(".")

def heading_chunks(text, max_words=150, min_words=25):
    """Split at headings; merge tiny sections forward; split oversized sections with fixed windows."""
    sections, cur_h, cur = [], "", []
    for line in text.splitlines():
        if line.strip() and is_heading(line.strip()):
            if any(l.strip() for l in cur):
                sections.append((cur_h, "\n".join(cur).strip()))
            cur_h, cur = line.strip().rstrip(":"), []
        else:
            cur.append(line)
    if any(l.strip() for l in cur):
        sections.append((cur_h, "\n".join(cur).strip()))
    out, carry, carry_h = [], "", []
    for i, (h, t) in enumerate(sections):
        if len(t.split()) < min_words and i < len(sections) - 1:        # too small: glue onto the next section
            carry += (f"{h}\n" if h else "") + t + "\n"; carry_h += [h] if h else []
            continue
        t = (carry + t).strip(); h = " + ".join(carry_h + ([h] if h else [])); carry, carry_h = "", []
        if len(t.split()) > max_words * 1.5:
            out += [(h, piece) for _, piece in fixed_chunks(t, max_words, 0.15)]
        else:
            out.append((h, t))
    if carry:
        out.append(("", carry.strip()))
    return out

d = docs.loc["D36"]
for name, fn in [("fixed(120)", fixed_chunks), ("paragraph(120)", paragraph_chunks), ("heading-aware", heading_chunks)]:
    ch = fn(d.body)
    print(f"{name:15} -> {len(ch):2d} chunks | words per chunk: {[len(t.split()) for _, t in ch]}")
print("\nHeading-aware sections found in D36 (lessons learned, 459 words):")
print([h for h, _ in heading_chunks(d.body)])

### Building the chunk table (with provenance and a *contextual header*)
Each chunk is stored with its metadata. For *embedding and BM25* we prepend a short **contextual header** - `document title [policy id] > section heading` - to the chunk text.
Why? A chunk saying *"Within 30 calendar days"* is meaningless alone; with its header it knows it belongs to the *Travel and Expense Policy 2024 > Deadline*. It is a cheap trick that often beats fancier fixes
(LLM-generated "contextual retrieval" is a more sophisticated version of the same idea).

We then compare strategies using **vector search over chunks, rolled up to documents** (best chunk wins), through our shared `evaluate` harness - plus one extra "answer-in-context" check:
*does the text we would hand to a reader/LLM actually contain the answer?*

In [ ]:
def make_chunks(strategy_fn, **kw) -> pd.DataFrame:
    rows = []
    for d, r in docs.iterrows():
        for j, (section, text) in enumerate(strategy_fn(r.body, **kw)):
            pid = f" [{r.policy_id}]" if pd.notna(r.get("policy_id")) else ""
            header = f"{r.title}{pid}" + (f" > {section}" if section else "")
            rows.append(dict(chunk_id=f"{d}#{j}", doc_id=d, title=r.title, department=r.department, date=r.date,
                             sensitivity=r.sensitivity, policy_id=r.get("policy_id"), section=section, text=text,
                             embed_text=f"{header}\n{text}", n_words=len(text.split())))
    return pd.DataFrame(rows)

def whole_doc(text):                       # baseline: 1 chunk per document
    return [("", text)]

def aggregate_to_docs(chunk_doc_ids, scores, k=10):
    """Chunk scores -> document ranking: a document scores as its best chunk (max-pooling)."""
    out, seen = [], set()
    for i in np.argsort(-scores):
        if not np.isfinite(scores[i]):
            break
        d = chunk_doc_ids[i]
        if d not in seen:
            seen.add(d); out.append((d, float(scores[i])))
            if len(out) == k:
                break
    return out

VARIANTS = {}                               # name -> (chunk_df, embedding matrix)
def build_variant(name, strategy_fn, **kw):
    t = time.time()
    CH = make_chunks(strategy_fn, **kw)
    E = EMB.encode(CH.embed_text.tolist(), cache=True)
    VARIANTS[name] = (CH, E)
    print(f"  built {name:22} {len(CH):4d} chunks, avg {CH.n_words.mean():5.0f} words  ({time.time() - t:.1f}s)")

print("Embedding every chunking variant (cached on disk after the first run)...")
build_variant("whole document", whole_doc)
for sz in (40, 80, 150, 300):
    build_variant(f"fixed {sz}w", fixed_chunks, size=sz, overlap=0.2)
build_variant("paragraph ~120w", paragraph_chunks, target=120)
build_variant("heading-aware", heading_chunks, max_words=150)

In [ ]:
# ---- "Answer-in-context" check: does the retrieved chunk actually contain the answer sentence? ----------
ANSWER_KEYS = {
    "can I work from Spain for a month": "20 working days",
    "how do I get my money back for a client dinner": "GBP 120 per head",
    "someone stole my phone, what should I do": "within one hour",
    "how much time off do I get when my baby is born": "26 weeks",
    "how often do I have to come into the office": "three days",
    "the building is flooded and nobody can get in, what is the plan": "within 60 minutes",
    "how long do we hold on to old project files": "7 years",
    "I got a new handset and can't sign in any more": "temporary access pass",
    "a customer asked us to delete everything we hold about them": "one business day",
    "who has to sign off if I want to buy software for my team": "10,000",
}

def chunk_vec_search(CH, E):
    ids = CH.doc_id.tolist()
    return lambda q, k=10: aggregate_to_docs(ids, E @ embed_query(q), k)

def answer_in_context(CH, E, k):
    """Share of questions whose top-k chunk texts contain the answer phrase; and avg words handed to the reader."""
    hits, words = [], []
    for q, key in ANSWER_KEYS.items():
        top = np.argsort(-(E @ embed_query(q)))[:k]
        hits.append(any(key.lower() in CH.text.iloc[i].lower() for i in top))
        words.append(CH.n_words.iloc[top].sum())
    return np.mean(hits), np.mean(words)

rows = []
for name, (CH, E) in VARIANTS.items():
    res = evaluate(chunk_vec_search(CH, E), name=name)
    a1, w1 = answer_in_context(CH, E, 1); a3, w3 = answer_in_context(CH, E, 3)
    rows.append({"strategy": name, "chunks": len(CH), "avg words": CH.n_words.mean(),
                 "Recall@5": res.loc["ALL (answerable)", "Recall@5"], "nDCG@5": res.loc["ALL (answerable)", "nDCG@5"],
                 "nDCG@5 semantic": res.loc["semantic", "nDCG@5"], "answer@1": a1, "answer@3": a3, "words in top-3": w3})
chunk_cmp = pd.DataFrame(rows).set_index("strategy")
display(chunk_cmp)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
fixed_rows = chunk_cmp.loc[[n for n in chunk_cmp.index if n.startswith("fixed")]]
sizes = [int(n.split()[1][:-1]) for n in fixed_rows.index]
ax[0].plot(sizes, fixed_rows["nDCG@5"], "o-", color=TEAL, label="nDCG@5 (all)")
ax[0].plot(sizes, fixed_rows["answer@1"], "s-", color=CORAL, label="answer in top-1 chunk")
ax[0].axhline(chunk_cmp.loc["whole document", "nDCG@5"], color=INK, ls=":", label="whole-document nDCG@5")
ax[0].set(title="Fixed-size chunking: size vs quality", xlabel="chunk size (words)", ylim=(0, 1.05)); ax[0].legend(frameon=False, fontsize=8)
ax[1].scatter(chunk_cmp["words in top-3"], chunk_cmp["answer@3"], s=90, color=TEAL, zorder=3)
for n, r in chunk_cmp.iterrows():
    ax[1].annotate(n, (r["words in top-3"], r["answer@3"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
ax[1].set(title="Context cost vs answer coverage (top-3 chunks)", xlabel="words handed to the reader/LLM", ylabel="answer@3")
plt.tight_layout(); plt.savefig(ART / "chunking.png"); plt.show()

**What to take away** (numbers are for the neural model; with the LSA fallback they will differ - the *method* does not):
* **A retrieval-only metric can pick the wrong winner.** Tiny 40-word chunks post the best document-level nDCG (precise matches!) - yet the answer sentence is rarely *inside* the top chunk (look at `answer@1`). Whole documents do the opposite: the answer is in there, but you hand the reader ~600 words to find it. The sweet spot depends on the corpus and on what consumes the chunks (a human skimming snippets? an LLM with a token budget?). **You find it with an evaluation set that measures what you care about, not by folklore** ("512 tokens, 10% overlap" is a starting point, not a law).
* **29 queries is a small sample:** differences of +/-0.02-0.03 nDCG between neighbours (e.g. fixed-150 vs heading-aware) are noise. Don't crown a winner on a gap you couldn't defend statistically - choose on the *other* axes too: context size, citation readability, robustness.
* Structure-aware chunking (heading-aware) gives short, *labelled*, self-contained chunks: answer coverage similar to whole documents at a fraction of the context, and readable citations ("Information Security Policy > 5. LOSS OR THEFT").
* Chunking is **upstream of everything**: a bad chunker can't be repaired by a better model or a bigger LLM. It is the most underrated lever in RAG systems.

Let's look at what the reader gets for the "stolen phone" question under three strategies:

In [ ]:
q = "someone stole my phone, what should I do"
for name in ["whole document", "fixed 150w", "heading-aware"]:
    CH, E = VARIANTS[name]
    i = int(np.argmax(E @ embed_query(q)))
    r = CH.iloc[i]
    print(f"--- {name} | top chunk {r.chunk_id} ({r.n_words} words) | cos = {float((E @ embed_query(q))[i]):.3f}")
    print(textwrap.shorten(r.text.replace("\n", " "), 420, placeholder=" ..."))
    print()

#### Exercise 5 - Write a sentence-window chunker (6 min)
Implement `sentence_window_chunks(text, window=3, stride=2)`: split into sentences, then emit overlapping windows of `window` sentences advancing by `stride`.
It must return the same shape as the others: `[("", chunk_text), ...]`. Then plug it into `build_variant` and compare in the table.
*(Production variant: embed single sentences but return the +/- 1 neighbours - "small-to-big retrieval".)*

In [ ]:
def sentence_window_chunks(text, window=3, stride=2):
    r"""TODO: return [("", " ".join(sentences[i:i+window])) ...] stepping by `stride`.
    Hint: sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+|\n+', text) if s.strip()]"""
    return [("", text)]

### Decision
We standardise on **heading-aware chunks (max 150 words) with a contextual header** for the rest of the lab. The metrics above say it is *competitive*, not dominant - we choose it for the combination of
good answer coverage in the top-3, a small context footprint, and **citations a human can verify** (a section title beats "characters 1,140-1,820").
If your corpus is scanned PDFs without structure, you'd pick paragraph or fixed-size and invest in layout parsing first - and you'd re-run this comparison on *your* data.

## 6. Vector search - exact first, then approximate

### Exact (brute-force) k-NN: one matrix multiply
With unit-length vectors, similarity to *every* chunk is a single matrix-vector product `E @ q`; take the top k. No index, no tuning, 100% exact recall.
For our ~150 chunks this takes microseconds. For 1M chunks x 384 dims it's ~0.4 GFLOP per query - still only a few tens of milliseconds on a laptop.
**Architect's rule of thumb: below ~100k-1M vectors, exact search is often the right answer.** You don't need a "vector database" to get started - just discipline about metadata and evaluation.

In [ ]:
CH, CH_EMB = VARIANTS["heading-aware"]
CH_IDS = CH.doc_id.tolist()
print(f"Production index: {len(CH)} chunks x {CH_EMB.shape[1]} dims = {CH_EMB.nbytes / 1024:.0f} KiB (float32)")

class FlatIndex:
    """Exact k-NN by brute force. mask (bool array) lets us pre-filter by metadata."""
    def __init__(self, X):
        self.X = np.ascontiguousarray(X, dtype=np.float32)
    def search(self, q, k=5, mask=None):
        s = self.X @ q                                          # (N,d) @ (d,) -> (N,) cosine scores
        if mask is not None:
            s = np.where(mask, s, -np.inf)
        k = min(k, len(s) - 1)
        idx = np.argpartition(-s, k)[:k]                        # O(N) partial sort, not O(N log N)
        idx = idx[np.argsort(-s[idx])]
        keep = np.isfinite(s[idx])
        return idx[keep], s[idx][keep]

flat = FlatIndex(CH_EMB)
q = "someone stole my phone, what should I do"
idx, sc = flat.search(embed_query(q), k=3)
for i, s in zip(idx, sc):
    print(f"{s:.3f}  {CH.chunk_id.iloc[i]:8} {CH.title.iloc[i][:40]:40} | {CH.section.iloc[i]}")

t = time.perf_counter()
for _ in range(1000):
    flat.search(embed_query(q), k=5)
print(f"\n{(time.perf_counter() - t):.3f} ms per query on {len(CH)} chunks (query embedding cached)")

### Why *approximate* nearest neighbours exist
Exact search is O(N): double the data, double the time. At 100M vectors that's too slow and too memory-hungry for interactive latency. **ANN indexes trade a little recall for a lot of speed.** The big families:

* **IVF (inverted file):** cluster the vectors with k-means into `nlist` cells. At query time, find the `nprobe` nearest centroids and scan only those cells. *Think: a supermarket - you walk to the right aisles instead of checking every shelf.*
* **HNSW (hierarchical navigable small world):** a multi-layer proximity graph. Start at the top sparse layer, greedily hop toward the query, drop down a layer, repeat. Very fast and high recall, but graph lives in RAM and updates/deletes are fiddlier. *Think: a "six degrees of separation" network with motorways on top and side streets at the bottom.*
* **PQ (product quantization):** compress each vector into a few bytes (split into sub-vectors, store codebook IDs). 10-50x less memory, at some accuracy cost. Often combined as IVF-PQ.
* **LSH (locality-sensitive hashing):** random hyperplanes hash similar vectors into the same bucket - elegant theory, rarely the production winner today.

Below we build IVF **by hand** (spherical k-means + inverted lists) on a synthetically enlarged corpus of **100,000** vectors made by mixing and perturbing our real chunk embeddings, and measure the trade-off: recall@10 vs latency as we raise `nprobe`.

In [ ]:
from scipy import sparse

N_BIG, D = 100_000, CH_EMB.shape[1]
def synth_vectors(n, base, rng, noise=0.5):
    """Random convex mixtures of 3 real chunk vectors + Gaussian noise, re-normalised. (Keeps realistic geometry.)"""
    out = np.empty((n, base.shape[1]), dtype=np.float32)
    for s in range(0, n, 10_000):
        m = min(10_000, n - s)
        pick = rng.integers(0, len(base), size=(m, 3))
        w = rng.dirichlet([1.0, 1.0, 1.0], size=m).astype(np.float32)
        mix = (base[pick] * w[:, :, None]).sum(1) + noise * rng.standard_normal((m, base.shape[1])).astype(np.float32) / np.sqrt(base.shape[1])
        out[s:s + m] = mix / np.linalg.norm(mix, axis=1, keepdims=True)
    return out

t = time.time()
BIG = synth_vectors(N_BIG, CH_EMB, np.random.default_rng(SEED))
QRY = synth_vectors(300, CH_EMB, np.random.default_rng(SEED + 1))
print(f"synthetic corpus: {BIG.shape} ({BIG.nbytes / 1e6:.0f} MB) generated in {time.time() - t:.1f}s")

class IVFIndex:
    """Inverted-file index: spherical k-means centroids + contiguous inverted lists."""
    def __init__(self, nlist=256, iters=8, seed=SEED):
        self.nlist, self.iters, self.seed = nlist, iters, seed
    def fit(self, X):
        r = np.random.default_rng(self.seed)
        C = X[r.choice(len(X), self.nlist, replace=False)].copy()
        for _ in range(self.iters):                                         # Lloyd's algorithm on the sphere
            assign = np.argmax(X @ C.T, axis=1)
            M = sparse.csr_matrix((np.ones(len(X), np.float32), (assign, np.arange(len(X)))), shape=(self.nlist, len(X)))
            C_new = np.asarray(M @ X)
            empty = np.linalg.norm(C_new, axis=1) < 1e-9
            C_new[empty] = X[r.choice(len(X), empty.sum(), replace=False)]   # re-seed empty clusters
            C = C_new / np.linalg.norm(C_new, axis=1, keepdims=True)
        assign = np.argmax(X @ C.T, axis=1)
        order = np.argsort(assign, kind="stable")                           # group members of each cell contiguously
        self.C, self.X, self.ids = C.astype(np.float32), np.ascontiguousarray(X[order]), order
        self.offsets = np.searchsorted(assign[order], np.arange(self.nlist + 1))
        return self
    def search(self, q, k=10, nprobe=8):
        cells = np.argpartition(-(self.C @ q), nprobe)[:nprobe]             # which aisles to visit
        scores, ids = [], []
        for c in cells:
            lo, hi = self.offsets[c], self.offsets[c + 1]
            scores.append(self.X[lo:hi] @ q); ids.append(self.ids[lo:hi])
        s, i = np.concatenate(scores), np.concatenate(ids)
        top = np.argpartition(-s, k)[:k]
        top = top[np.argsort(-s[top])]
        return i[top], s[top]

t = time.time()
ivf = IVFIndex(nlist=256, iters=6).fit(BIG)
print(f"IVF built in {time.time() - t:.1f}s | {ivf.nlist} cells, avg {len(BIG) / ivf.nlist:.0f} vectors per cell")

In [ ]:
# ---- Benchmark: exact vs IVF at several nprobe values (300 queries, recall@10 against exact truth) ----------
flat_big = FlatIndex(BIG)
truth = [set(flat_big.search(q, 10)[0]) for q in QRY]

def bench(fn, repeats=3):
    """Average per-query latency (ms); best of `repeats` runs to damp scheduler noise. Also returns the last results."""
    best = float("inf")
    for _ in range(repeats):
        t = time.perf_counter()
        res = [fn(q) for q in QRY]
        best = min(best, (time.perf_counter() - t) / len(QRY) * 1000)
    return best, res

lat_flat, _ = bench(lambda q: flat_big.search(q, 10))
rows = [{"index": "exact (brute force)", "nprobe": "-", "latency ms": lat_flat, "recall@10": 1.0, "scanned %": 100.0}]
for npr in (1, 2, 4, 8, 16, 32, 64):
    lat, res = bench(lambda q: ivf.search(q, 10, npr))
    rec = np.mean([len(set(r[0]) & t_) / 10 for r, t_ in zip(res, truth)])
    scanned = 100 * npr / ivf.nlist
    rows.append({"index": "IVF", "nprobe": npr, "latency ms": lat, "recall@10": rec, "scanned %": scanned})
ann = pd.DataFrame(rows)
display(ann)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ivf_rows = ann[ann["index"] == "IVF"]
ax[0].plot(ivf_rows["latency ms"], ivf_rows["recall@10"], "o-", color=TEAL, label="IVF (nprobe sweep)")
for _, r in ivf_rows.iterrows():
    ax[0].annotate(f"nprobe={r['nprobe']}", (r["latency ms"], r["recall@10"]), fontsize=7, xytext=(3, -9), textcoords="offset points")
ax[0].axvline(lat_flat, color=CORAL, ls="--", label=f"exact search = {lat_flat:.1f} ms (recall 1.0)")
ax[0].set(xlabel="latency per query (ms)", ylabel="recall@10 vs exact", title="ANN trade-off: recall vs latency"); ax[0].legend(frameon=False, fontsize=8)

sizes = [1_000, 5_000, 20_000, 50_000, 100_000]
lat_n = []
for n in sizes:                                          # exact-search latency as the collection grows
    sub = FlatIndex(BIG[:n])
    t0 = time.perf_counter()
    for qv in QRY[:50]:
        sub.search(qv, 10)
    lat_n.append((time.perf_counter() - t0) / 50 * 1000)
ax[1].plot(sizes, lat_n, "o-", color=CORAL); ax[1].set(xscale="log", xlabel="number of vectors", ylabel="exact search latency (ms)",
                                                       title="Brute force scales linearly with N")
plt.tight_layout(); plt.savefig(ART / "ann.png"); plt.show()

best = ivf_rows[ivf_rows["recall@10"] >= 0.95].head(1)
if len(best):
    b = best.iloc[0]
    print(f"First setting with recall >= 0.95: nprobe={b['nprobe']} -> {lat_flat / b['latency ms']:.1f}x faster than exact, scanning only {b['scanned %']:.1f}% of vectors.")
print("Dial `nprobe` up -> recall rises toward 1.0 and latency rises. That single knob is the whole ANN story (HNSW's equivalent is `ef_search`).")

### Choosing a vector store - the architect's cheat sheet
You now know what these products are *doing inside*: some flavour of exact scan, IVF, HNSW, PQ - plus metadata filtering, persistence, replication, and an API. Differences are mostly about **operations and ecosystem**, not magic.

| option | what it is | pick it when | watch out for |
|---|---|---|---|
| **NumPy / FAISS (flat)** | library; exact or ANN in-process | prototypes, < ~1M vectors, you control the host | no server, filtering/persistence are DIY |
| **FAISS (IVF/HNSW/PQ)** | Meta's ANN library | you need raw speed/memory control, batch/offline | a library, not a database: no ACLs, CRUD is clunky |
| **pgvector (PostgreSQL)** | vector type + HNSW/IVF in Postgres | you already run Postgres; need SQL joins + transactions + row-level security together | scale-out limits at very large N; tune index builds |
| **Azure AI Search** | managed search: BM25 + vector + semantic ranker + security filters | Microsoft estate; want hybrid + ACL trimming + less ops | feature coupling to Azure; index schema changes need planning |
| **OpenSearch / Elasticsearch** | search engine with kNN (HNSW) + mature BM25 | you already run it; hybrid is first-class; logs + search together | cluster operations; JVM tuning |
| **Pinecone** | fully managed vector DB | small team, want zero-ops, elastic scale | vendor lock-in; data residency questions |
| **Weaviate** | open-source vector DB with hybrid + modules | want built-in hybrid and schema; self-host or managed | operational maturity at your scale: test it |
| **Qdrant** | open-source vector DB (Rust), strong payload filtering | heavy metadata filtering, self-hosted performance | smaller ecosystem than the giants |
| **Milvus** | distributed open-source vector DB | very large (100M-1B+) collections, GPU/disk indexes | complexity: multiple moving parts |
| **Chroma** | lightweight embedded/dev vector store | notebooks, demos, local apps | production hardening is on you |

**Selection questions to ask a client:** How many vectors now and in 2 years? Do you need **hybrid** and **row-level security**? Who operates it? Data residency / private networking? Update rate (freshness SLA)? Can you tolerate re-indexing? Do you already run Postgres / OpenSearch / Azure Search? *(Pick the boring option you already operate unless the numbers say otherwise.)*

## 7. Metadata filtering, freshness and access control

Pure similarity search is *context-blind*. Real enterprises need three more things, and all three come from **metadata we kept on every chunk**:

1. **Scoping filters** - department, date range, document type ("only IT how-tos").
2. **Freshness / versioning** - the 2022 and 2024 parental-leave policies are *nearly identical text*, so their embeddings are neighbours. Similarity can't tell you which is current; **metadata can** (`policy_id` + `date`). Returning the superseded policy is not a search bug, it's a *compliance incident*.
3. **Security trimming (ACLs)** - users must only ever see what they are entitled to see. Our simplified model: `sensitivity` label vs. role (`contractor` -> public; `employee` -> public+internal; `security_lead` -> everything).

> **The golden rule: filter BEFORE ranking (pre-filter), never after.** Post-filtering (rank first, delete forbidden results afterwards) leaks information into everything that touched the un-trimmed list - snippets, answer generation, "did you mean", result counts, logs - and silently returns fewer than `k` results. We will demonstrate the leak, not just assert it.

In [ ]:
ROLE_ACCESS = {"contractor": {"public"}, "employee": {"public", "internal"}, "security_lead": {"public", "internal", "confidential"}}

def stale_doc_ids() -> set:
    """Documents superseded by a newer document with the same policy_id."""
    d = docs[docs.policy_id.notna()]
    newest = d.groupby("policy_id").date.transform("max")
    return set(d.doc_id[d.date < newest])
STALE = stale_doc_ids()
print("Superseded documents detected from metadata:", sorted(STALE))

def make_mask(CH, role="security_lead", department=None, since=None, latest_only=False):
    """Boolean mask over chunks: True = this chunk is allowed to compete for ranking."""
    m = CH.sensitivity.isin(ROLE_ACCESS[role]).to_numpy().copy()      # security trimming
    if department:  m &= (CH.department == department).values          # scoping filter
    if since:       m &= (CH.date >= pd.Timestamp(since)).values       # date filter
    if latest_only: m &= ~CH.doc_id.isin(STALE).values                 # freshness: drop superseded versions
    return m

def vec_search_filtered(q, k=5, **filters):
    s = np.where(make_mask(CH, **filters), CH_EMB @ embed_query(q), -np.inf)       # PRE-filter: masked chunks can't rank
    return aggregate_to_docs(CH_IDS, s, k)

def show(res, n=4):
    return [f"{d}({docs.date[d].year}, {docs.sensitivity[d][:3]})  {sc:.3f}" for d, sc in res[:n]]

q = "how much time off do I get when my baby is born"
print(f"\nQ: {q!r}")
print("  no filter          :", show(vec_search_filtered(q)))
print("  since 2024-01-01   :", show(vec_search_filtered(q, since="2024-01-01")))
print("  latest_only (meta) :", show(vec_search_filtered(q, latest_only=True)))
q = "I forgot how to sign in"
print(f"\nQ: {q!r}")
print("  no filter          :", show(vec_search_filtered(q)))
print("  department=Security:", show(vec_search_filtered(q, department="Security")))
print("  department=IT      :", show(vec_search_filtered(q, department="IT")))

In [ ]:
# ---- Does freshness filtering fix the gold queries designed around it? -----------------------------------
fresh_qs = [e for e in EVAL if "FRESHNESS" in e.get("note", "") or e["query"].startswith("POL-FIN-006") or e["query"] == "POL-HR-017"]
rows = []
for e in fresh_qs:
    for label, kw in [("no freshness rule", {}), ("latest_only", {"latest_only": True})]:
        ranked = [d for d, _ in vec_search_filtered(e["query"], k=5, **kw)]
        rows.append({"query": e["query"][:55], "gold": ",".join(e["relevant_doc_ids"]), "mode": label, "top-2": ranked[:2],
                     "nDCG@5": ndcg_at_k(ranked, e["relevant_doc_ids"], 5)})
display(pd.DataFrame(rows))
print("Freshness rules help the questions about CURRENT policy. But 'POL-HR-017' as a pure lookup lists both versions in the gold set:")
print("an auditor may legitimately want the old one. Design choice: default to latest, expose an explicit 'include history' toggle.")

#### Exercise 6 - A soft freshness boost (6 min)
A hard filter is blunt: sometimes an *older* document is still the best answer (nothing newer exists), so deleting it hurts. A softer alternative is a **recency prior**:
`final = score + weight * 0.5 ** (age_days / half_life_days)` - newer documents get a small bonus that fades with age.
Implement `recency_boost(scores, dates, as_of, half_life_days=365, weight=0.15)` and check that, for the "baby" question, the 2024 policy now ranks above the 2022 one **without** deleting anything.

In [ ]:
AS_OF = docs.date.max()          # "today" for the corpus (use datetime.now() in production)
def recency_boost(scores, dates, as_of=AS_OF, half_life_days=365, weight=0.15):
    """TODO: return scores + weight * 0.5 ** (age_in_days / half_life_days)
    Hint: age_days = (as_of - dates).dt.days.values  (dates is a pandas Series of Timestamps)"""
    return scores

### Security trimming - and the leak you get from doing it wrong
Two pipelines for the same user (an `employee`, who may not see `confidential` documents):

* **Naive (post-filter):** retrieve top-k over *everything*, assemble the "context", then remove forbidden results from the display.
* **Correct (pre-filter):** the mask is applied inside the scoring step, so forbidden chunks can never be retrieved, ranked, summarised or counted.

We track what the *context* contained (that's what a downstream LLM, snippet generator or log line would have seen).

In [ ]:
def naive_post_filter(q, role, k=5):
    context = vec_search_filtered(q, k)                         # ranked over EVERYTHING
    visible = [(d, s) for d, s in context if docs.sensitivity[d] in ROLE_ACCESS[role]]
    return context, visible

q, role = "an anonymous container exposed survey responses with email addresses to the internet", "employee"
context, visible = naive_post_filter(q, role)
print(f"Query: {q!r}  (role = {role})\n")
print("NAIVE  context handed to snippet/LLM stage:", [(d, docs.sensitivity[d]) for d, _ in context])
print("NAIVE  visible after deletion              :", [d for d, _ in visible], f"-> only {len(visible)} of 5 results")
print("CORRECT pre-filtered results               :", [(d, docs.sensitivity[d]) for d, _ in vec_search_filtered(q, 5, role=role)])
leaked = [d for d, _ in context if docs.sensitivity[d] not in ROLE_ACCESS[role]]
print(f"\nLEAK: the naive pipeline processed confidential documents {leaked} -> their text would have fed the summary/LLM, and the "
      f"short result list reveals that something relevant exists.")

rows = []
for role in ["contractor", "employee"]:
    leak, vis_naive, vis_correct = 0, [], []
    for e in EVAL:
        ctx, vis = naive_post_filter(e["query"], role)
        leak += any(docs.sensitivity[d] not in ROLE_ACCESS[role] for d, _ in ctx)
        vis_naive.append(len(vis)); vis_correct.append(len(vec_search_filtered(e["query"], 5, role=role)))
    rows.append({"role": role, "queries whose context leaked": f"{leak}/{len(EVAL)}",
                 "avg results (naive)": np.mean(vis_naive), "avg results (pre-filter)": np.mean(vis_correct)})
display(pd.DataFrame(rows))
print("Pre-filtering fixes both problems: no forbidden text is ever touched, and the user still gets a full page of allowed results.")

**Architect's note - how this works in real systems.** The mask above is what Azure AI Search calls *security filters*, what OpenSearch calls *document-level security*,
and what pgvector does with a `WHERE` clause or Postgres row-level security. Pitfalls to design for:
(1) ACLs change (people join groups, leave) - sync permissions *separately* from content and filter at query time with the user's **current** group IDs, not a snapshot baked into vectors;
(2) heavy filters + ANN indexes can cut recall (the graph walk wanders into forbidden regions) - test *filtered* recall explicitly, a very common production surprise;
(3) the filter label is only as good as the metadata - one wrongly-labelled `public` document is a leak. Treat label quality as a data-quality KPI.

## 8. Hybrid search: the best of both worlds

We now have two retrievers with complementary strengths:

* **BM25** nails exact tokens (`ERR-4021`, `POL-SEC-012`, product names), is fast, cheap, explainable - blind to paraphrase.
* **Dense vectors** understand paraphrase and intent - blurry on identifiers and rare jargon, and "approximately right" can be confidently wrong.

**Fusion** combines their rankings. Two classic recipes:

1. **Reciprocal Rank Fusion (RRF):** `score(d) = sum over retrievers of 1 / (k + rank_r(d))`, with k ~ 60. Uses only *ranks*, so you never have to make incomparable scores (BM25 of 14.2 vs cosine of 0.71) comparable. Robust, tuning-free; what Elasticsearch, OpenSearch, Azure AI Search and Weaviate offer out of the box.
2. **Weighted score fusion:** min-max normalise each score list to [0,1], then `alpha * dense + (1-alpha) * sparse`. More control, but sensitive to score distributions - and `alpha` needs tuning *on your eval set*.

Afterwards, an optional **re-ranker** (a cross-encoder reading query+passage *together*) re-orders the top ~20-50. It's slower but much more accurate than bi-encoder similarity. We don't have a cross-encoder offline here,
so we'll implement a clearly-labelled **lexical-overlap stand-in** to show where the stage sits in the pipeline - and to teach you to be sceptical of rerankers you haven't measured.

In [ ]:
# ---- Chunk-level retrievers: scores for every chunk, then aggregate to documents ---------------------------
bm25_ch = BM25().fit(CH.embed_text.tolist())

def s_bm25(q):  return bm25_ch.scores(q)
def s_vec(q):   return CH_EMB @ embed_query(q)

def rrf_fuse(score_arrays, k_rrf=60, depth=50):
    """Reciprocal Rank Fusion over several score arrays (higher = better). Zero/-inf scores don't vote."""
    fused = np.zeros(len(score_arrays[0]))
    for s in score_arrays:
        for rank, i in enumerate(np.argsort(-s)[:depth], start=1):
            if np.isfinite(s[i]) and s[i] != 0:
                fused[i] += 1.0 / (k_rrf + rank)
    return fused

def minmax(s):
    s = np.asarray(s, dtype=float); ok = np.isfinite(s)
    lo, hi = s[ok].min(), s[ok].max()
    return np.where(ok, (s - lo) / (hi - lo + 1e-12), 0.0)

def s_rrf(q):                return rrf_fuse([s_bm25(q), s_vec(q)])
def s_weighted(q, alpha=0.5): return alpha * minmax(s_vec(q)) + (1 - alpha) * minmax(s_bm25(q))

def as_doc_search(score_fn):
    return lambda q, k=10: aggregate_to_docs(CH_IDS, score_fn(q), k)

score_system("BM25 (chunks)",        as_doc_search(s_bm25), show=False)
score_system("Dense (chunks)",       as_doc_search(s_vec), show=False)
score_system("Hybrid RRF",           as_doc_search(s_rrf), show=False)
score_system("Hybrid weighted a=.5", as_doc_search(s_weighted), show=False)

In [ ]:
# ---- STAND-IN reranker (NOT a cross-encoder): lexical overlap between query and candidate chunk ----------
def lexical_rerank_scores(q, base_scores, depth=20, w_overlap=0.4):
    """Re-score the top `depth` candidates: 60% normalised fused score + 40% share of query terms found in the chunk.
    A real cross-encoder (e.g. ms-marco MiniLM, bge-reranker, Cohere Rerank) would read (query, passage) jointly instead."""
    qt = set(tokenize(q))
    out = np.full(len(base_scores), -np.inf)
    top = np.argsort(-base_scores)[:depth]
    base = minmax(base_scores[top])
    for b, i in zip(base, top):
        ct = set(tokenize(CH.embed_text.iloc[i]))
        overlap = len(qt & ct) / max(1, len(qt))
        out[i] = (1 - w_overlap) * b + w_overlap * overlap
    return out

score_system("Hybrid RRF + lexical rerank (stand-in)", as_doc_search(lambda q: lexical_rerank_scores(q, s_rrf(q))), show=False)

# ---- The leaderboard: nDCG@5 by query type ------------------------------------------------------------
def leaderboard_table(metric="nDCG@5", names=None):
    names = names or list(LEADERBOARD)
    rows = {}
    for n in names:
        t = LEADERBOARD[n]
        rows[n] = {typ: t.loc[typ, metric] for typ in ["keyword", "semantic", "hybrid"]} | {"ALL": t.loc["ALL (answerable)", metric],
                                                                                           "MRR (ALL)": t.loc["ALL (answerable)", "MRR"],
                                                                                           "Recall@5 (ALL)": t.loc["ALL (answerable)", "Recall@5"]}
    return pd.DataFrame(rows).T

LB = leaderboard_table()
display(LB.sort_values("ALL", ascending=False))

core = ["BM25 (chunks)", "Dense (chunks)", "Hybrid RRF", "Hybrid weighted a=.5"]
fig, ax = plt.subplots(figsize=(10.5, 4.2))
types = ["keyword", "semantic", "hybrid", "ALL"]; w = 0.2
for j, (name, col) in enumerate(zip(core, [GOLD, CORAL, TEAL, INK])):
    vals = [LB.loc[name, t] for t in types]
    bars = ax.bar(np.arange(len(types)) + (j - 1.5) * w, vals, w, label=name, color=col)
    for b_, v in zip(bars, vals):
        ax.text(b_.get_x() + b_.get_width() / 2, v + 0.01, f"{v:.2f}", ha="center", fontsize=7)
ax.set_xticks(range(len(types))); ax.set_xticklabels(["keyword queries", "semantic queries", "hybrid queries", "ALL answerable"])
ax.set(ylabel="nDCG@5", ylim=(0, 1.12), title="Leaderboard: no single retriever wins everywhere"); ax.legend(frameon=False, ncol=2, fontsize=8)
plt.tight_layout(); plt.savefig(ART / "leaderboard.png"); plt.show()

**Reading the leaderboard.** Look for the *shape*, not just the top number: the lexical retriever owns the `keyword` bars, the dense retriever owns `semantic`, and fusion should be near the best of both on every group - that is the entire business case for hybrid.
Also note the stand-in reranker: if it doesn't help, **that is a valid result** - never ship a stage you haven't measured. (A real cross-encoder usually helps most when the first stage has good *recall* but mediocre *ordering*.)

#### Exercise 7 - Tune the fusion weight (5 min)
For weighted fusion, sweep `alpha` from 0 (BM25 only) to 1 (dense only) and plot overall nDCG@5. Where is the optimum? Is the curve flat or peaky?
*(Caveat to discuss: tuning on the same 33 queries you report on is optimistic. Real projects hold out a test split.)*

In [ ]:
# TODO: for alpha in np.linspace(0, 1, 11): evaluate as_doc_search(lambda q: s_weighted(q, alpha)) and store the ALL nDCG@5.
# Hint: res = evaluate(fn); res.loc["ALL (answerable)", "nDCG@5"]
alphas, ndcgs = [], []

## 9. Putting it together: a `Search` service

Time to package everything into one object with a clean contract - the thing you would wrap in a REST endpoint and hand to the application team:

```
Search.query(text, k, role, department, since, latest_only, mode) -> Response
    Response.hits[]      rank, doc_id, chunk_id, title, section, metadata, score, snippet (with highlights), "why" explanation
    Response.confident   False => show "no confident answer" instead of a confidently wrong one
    Response.latency_ms  per-stage timings
```

Design decisions baked in (each one is a lesson from earlier sections):
1. **Pre-filter** by role / department / date / freshness *inside* scoring.
2. **Hybrid** BM25 + dense, fused with RRF, over heading-aware chunks with provenance.
3. **Explainability:** matched keywords highlighted, ranks from each retriever shown - when a stakeholder asks "why is *this* the top result?", you can answer.
4. **Abstention:** if the best evidence is weak, say so. A search box that always returns something teaches users to distrust it.

For the abstention rule we need a **confidence signal**. RRF scores are about ranks, not quality (the #1 result always gets ~0.033), so they cannot tell a good answer from a bad one. Candidates:

* the **best dense cosine** - intuitive, but bge-style models squeeze most cosines into a narrow 0.5-0.8 band, so "0.66" can be a perfect match or a lucky near-miss;
* the **margin** - how far the best document stands out from the 5th best. A real answer is a *peak*; a question the corpus can't answer produces a flat plateau of mediocre matches;
* an **exact identifier match** (`ERR-4021`-style tokens) - decisive evidence that cosine/margin underrate.

We will not guess which works: we will *measure* it on the gold set (which is why it contains `no_answer` queries).

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Hit:
    rank: int; doc_id: str; chunk_id: str; title: str; section: str; department: str; date: str; sensitivity: str
    score: float; cosine: float; vec_rank: int; bm25_rank: int | None; matched: list; snippet: str; why: str; text: str

@dataclass
class Response:
    query: str; hits: list; confident: bool; reason: str; top_cosine: float; margin: float; exact_id: bool; latency_ms: dict = field(default_factory=dict)

class Search:
    def __init__(self, chunks, embeddings, embedder, threshold=None, rrf_k=60):
        self.CH, self.E, self.emb = chunks.reset_index(drop=True), embeddings, embedder
        self.bm25 = BM25().fit(self.CH.embed_text.tolist())
        self.chunk_tokens = [set(tokenize(t)) for t in self.CH.embed_text]
        self.stale, self.threshold, self.rrf_k = stale_doc_ids(), threshold, rrf_k

    # -- helpers -------------------------------------------------------------------------------------------
    def _mask(self, role, department, since, latest_only):
        m = self.CH.sensitivity.isin(ROLE_ACCESS[role]).to_numpy().copy()
        if department:  m &= (self.CH.department == department).values
        if since:       m &= (self.CH.date >= pd.Timestamp(since)).values
        if latest_only: m &= ~self.CH.doc_id.isin(self.stale).values
        return m

    @staticmethod
    def highlight(text, qterms):
        """Wrap words that share a (stemmed) token with the query in << >>."""
        def repl(m):
            return f"<<{m.group(0)}>>" if any(t in qterms for t in tokenize(m.group(0))) else m.group(0)
        return re.sub(r"[A-Za-z0-9][A-Za-z0-9_-]*", repl, text)

    def _snippet(self, text, qterms, width=230):
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+|\n+", text) if s.strip()] or [text]
        best = max(sents, key=lambda s: len(qterms & set(tokenize(s))))           # sentence with most query terms
        return self.highlight(textwrap.shorten(best, width, placeholder=" ..."), qterms)

    # -- the public API ------------------------------------------------------------------------------------
    def query(self, q, k=5, role="employee", department=None, since=None, latest_only=True, mode="hybrid", use_threshold=True):
        lat, t0 = {}, time.perf_counter()
        mask = self._mask(role, department, since, latest_only)
        qv = self.emb.encode([q], is_query=True)[0];                                         lat["embed"] = time.perf_counter() - t0
        t1 = time.perf_counter()
        cos = np.where(mask, self.E @ qv, -np.inf);                                          lat["dense"] = time.perf_counter() - t1
        t2 = time.perf_counter()
        bm = np.where(mask, self.bm25.scores(q), 0.0);                                       lat["bm25"] = time.perf_counter() - t2
        t3 = time.perf_counter()
        fused = {"hybrid": lambda: rrf_fuse([bm, cos], self.rrf_k), "dense": lambda: cos, "bm25": lambda: bm}[mode]()
        fused = np.where(mask, fused, -np.inf)
        ranked = aggregate_to_docs(self.CH.doc_id.tolist(), fused, k)                       # best chunk per document
        lat["fuse"] = time.perf_counter() - t3
        # rank bookkeeping for the "why" explanation
        vec_rank = {i: r for r, i in enumerate(np.argsort(-cos), 1)}
        bm_rank = {i: r for r, i in enumerate([j for j in np.argsort(-bm) if bm[j] > 0], 1)}
        qterms = set(tokenize(q))
        best_chunk = {}
        for i in np.argsort(-fused):
            if np.isfinite(fused[i]) and self.CH.doc_id.iloc[i] not in best_chunk:
                best_chunk[self.CH.doc_id.iloc[i]] = i
        hits = []
        for r, (d, sc) in enumerate(ranked, 1):
            i = best_chunk[d]; row = self.CH.iloc[i]
            matched = sorted(t for t in qterms & self.chunk_tokens[i] if "-" in t or t.isdigit() or len(t) > 3)
            why = f"dense #{vec_rank[i]} (cos {cos[i]:.2f}) | BM25 #{bm_rank.get(i, '-')} | " + \
                  (f"shared terms: {', '.join(matched[:5])}" if matched else "pure semantic match (no shared keywords)")
            hits.append(Hit(r, d, row.chunk_id, row.title, row.section, row.department, str(row.date.date()), row.sensitivity, sc,
                            float(cos[i]), vec_rank[i], bm_rank.get(i), matched, self._snippet(row.text, qterms), why, row.text))
        # confidence signals: best cosine, margin (best doc vs 5th best doc), exact identifier hit (e.g. ERR-4021, POL-SEC-012)
        top_docs = [s for _, s in aggregate_to_docs(self.CH.doc_id.tolist(), cos, 5)]
        top_cos = top_docs[0] if top_docs else 0.0
        margin = top_docs[0] - top_docs[-1] if len(top_docs) > 1 else 0.0
        id_toks = [t for t in qterms if "-" in t and any(c.isdigit() for c in t)]
        exact = any(t in self.chunk_tokens[i] for t in id_toks for i in np.where(mask)[0])
        confident, reason = True, "ok"
        if use_threshold and self.threshold is not None and not exact and margin < self.threshold:
            confident, reason = False, f"flat score profile (margin {margin:.3f} < {self.threshold:.3f}) and no exact identifier match"
        lat["total"] = time.perf_counter() - t0
        return Response(q, hits, confident, reason, top_cos, margin, exact, {k_: v * 1000 for k_, v in lat.items()})

SEARCH = Search(CH, CH_EMB, EMB)                       # no threshold yet - we calibrate it next
r = SEARCH.query("can I work from Spain for a month", k=3)
print(f"confident={r.confident}  top cosine={r.top_cosine:.3f}  margin={r.margin:.3f}  latency={r.latency_ms['total']:.0f} ms")
for h in r.hits:
    print(f"#{h.rank} {h.doc_id} | {h.title[:60]} > {h.section}\n    {h.why}\n    {h.snippet}")

### Calibrating the "I don't know" threshold
We collect the three signals for every gold query and ask: *which one separates answerable from no-answer queries?* The standard yardstick is **ROC-AUC** (0.5 = coin flip, 1.0 = perfect ranking of answerable above no-answer).
Then we choose the cut-off that best balances the two error types:

* *False answer* (no answer exists, but we show results confidently) -> user trusts wrong info.
* *False abstain* (answer exists, but we say "I don't know") -> user frustrated, calls the helpdesk.

Which error is worse is a **business decision** (a legal-advice bot should abstain more eagerly than a lunch-menu bot). The curves make that conversation concrete.
*(With only 33 queries - and calibrating on the same queries we report on - treat the numbers as indicative; re-calibrate on real query logs, per scope: a heavily filtered search has fewer candidates and a different margin distribution.)*

In [ ]:
from sklearn.metrics import roc_auc_score

sig = []
for e in EVAL:
    r = SEARCH.query(e["query"], k=3, role="security_lead", latest_only=False, use_threshold=False)
    sig.append({"query": e["query"], "type": e["type"], "top_cosine": r.top_cosine, "margin": r.margin, "exact_id": r.exact_id})
sig = pd.DataFrame(sig)
is_ans = sig.type != "no_answer"
answerable, noans = sig[is_ans], sig[~is_ans]
print(f"ROC-AUC, answerable vs no-answer:  best cosine = {roc_auc_score(is_ans, sig.top_cosine):.2f}   |   margin = {roc_auc_score(is_ans, sig.margin):.2f}")

grid = np.round(np.arange(0, sig.margin.max() + 0.01, 0.0025), 4)
sweep = pd.DataFrame({
    "threshold": grid,
    "answers kept": [np.mean((answerable.margin >= t) | answerable.exact_id) for t in grid],       # exact-id bypasses the rule
    "no-answer rejected": [np.mean((noans.margin < t) & ~noans.exact_id) for t in grid],
})
sweep["balanced"] = (sweep["answers kept"] + sweep["no-answer rejected"]) / 2
best_bal = sweep.balanced.max()
plateau = sweep[sweep.balanced >= best_bal - 1e-9].threshold            # all thresholds that tie for best
THRESHOLD = float(round(plateau.median(), 4))                              # middle of the plateau = most robust
chosen = sweep.iloc[(sweep.threshold - THRESHOLD).abs().argmin()]
print(f"balanced accuracy peaks at {best_bal:.3f} for thresholds {plateau.min()}-{plateau.max()}; chosen margin threshold = {THRESHOLD}")
print(f"at that threshold: {chosen['answers kept']:.0%} of answerable queries kept, {chosen['no-answer rejected']:.0%} of no-answer queries rejected")

fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
for a_, col, title, thr in [(ax[0], "top_cosine", "Signal 1: best cosine (weak)", None), (ax[1], "margin", "Signal 2: margin top-1 vs top-5 (strong)", THRESHOLD)]:
    bins = np.linspace(sig[col].min() - 0.005, sig[col].max() + 0.005, 16)
    a_.hist(answerable[col], bins=bins, color=TEAL, alpha=0.85, label="answerable")
    a_.hist(noans[col], bins=bins, color=CORAL, alpha=0.9, label="no-answer")
    if thr: a_.axvline(thr, color=INK, ls="--", label=f"threshold {thr}")
    a_.set(title=title, xlabel=col, ylabel="queries"); a_.legend(frameon=False, fontsize=8)
ax[2].plot(sweep.threshold, sweep["answers kept"], color=TEAL, label="answerable kept")
ax[2].plot(sweep.threshold, sweep["no-answer rejected"], color=CORAL, label="no-answer rejected")
ax[2].axvline(THRESHOLD, color=INK, ls="--"); ax[2].set(xlabel="margin threshold", title="The trade-off you must choose"); ax[2].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(ART / "threshold.png"); plt.show()

print("\nNo-answer queries (should sit LOW):")
display(noans[["query", "top_cosine", "margin"]])
print("Answerable queries with the weakest margin (where we may wrongly abstain):")
display(answerable.nsmallest(5, "margin")[["query", "type", "top_cosine", "margin", "exact_id"]])
SEARCH.threshold = THRESHOLD

### Demo: the search app in action
Six showcase queries as a user would type them - paraphrases, an error code, a typo-ish question, and a no-answer. Watch the *explanations*, the highlighted terms (`<<like this>>`), the metadata, and the abstention. Each call is timed.

In [ ]:
def show_response(r, k=3):
    flag = "OK " if r.confident else "NO CONFIDENT ANSWER"
    print("=" * 110)
    print(f"Q: {r.query!r}   [{flag}]  cos {r.top_cosine:.2f}, margin {r.margin:.3f}{'  exact-id match' if r.exact_id else ''}  |  {r.latency_ms['total']:.1f} ms "
          f"(embed {r.latency_ms['embed']:.1f}, dense {r.latency_ms['dense']:.2f}, bm25 {r.latency_ms['bm25']:.2f})")
    if not r.confident:
        print(f"   -> I could not find a confident answer ({r.reason}). Closest (low confidence) candidates:")
    rows = [{"#": h.rank, "doc": h.doc_id, "title": textwrap.shorten(h.title, 46, placeholder="..."), "section": textwrap.shorten(h.section or "-", 24, placeholder="..."),
             "dept": h.department, "date": h.date, "label": h.sensitivity[:4], "rrf": round(h.score, 3), "cos": round(h.cosine, 2)} for h in r.hits[:k]]
    print(pd.DataFrame(rows).to_string(index=False))
    if r.hits and r.confident:
        print(f"   top snippet: {r.hits[0].snippet}")
        print(f"   why        : {r.hits[0].why}")

SHOWCASE = ["can I work from Spain for a month", "ERR-4021 on the VPN", "how much leave do new parents get",
            "someone stole my phone, what should I do", "who signs off a software purchase for my team", "what is the dress code for client meetings"]
SEARCH.query("warm-up", k=1)                      # first call pays one-off initialisation costs; exclude it from timings
for q_ in SHOWCASE:
    show_response(SEARCH.query(q_, k=3, role="employee"))

In [ ]:
# ---- Same service, different users: the contractor cannot see internal/confidential material -------------------
q_ = "what went wrong in the retail ERP migration and what should we do differently"
for role in ["security_lead", "employee", "contractor"]:
    r = SEARCH.query(q_, k=3, role=role)
    print(f"role={role:14} -> confident={r.confident!s:5} | top docs: {[(h.doc_id, h.sensitivity[:4]) for h in r.hits]}")

# ---- Scoring the finished service with the same harness + abstention scorecard ---------------------------------
svc_fn = lambda q: [(h.doc_id, h.score) for h in SEARCH.query(q, k=10, role="security_lead", latest_only=False, use_threshold=False).hits]
score_system("Search service (hybrid, RRF)", svc_fn, show=False)
decisions = pd.DataFrame([{"type": e["type"], "abstained": not SEARCH.query(e["query"], role="security_lead", latest_only=False).confident} for e in EVAL])
print("\nAbstention rate by query type (want ~0% for answerable types and 100% for no_answer):")
display(decisions.groupby("type").abstained.mean().to_frame("abstained"))

# ---- Latency profile: end-to-end per query, including live query embedding --------------------------------------
lats = pd.DataFrame([SEARCH.query(e["query"], role="employee").latency_ms for e in EVAL])
display(lats.describe(percentiles=[0.5, 0.95]).loc[["mean", "50%", "95%", "max"]].round(2))
print("Total latency is dominated by embedding the query (a neural forward pass); dense + BM25 + fusion over ~150 chunks are ~1 ms. At 10M chunks "
      "the dense step grows (ANN index!) while query embedding stays constant.")

## 10. Bridge to RAG (Day 3 teaser): from search results to a grounded prompt

Everything so far was **retrieval** - finding the right evidence. **Retrieval-Augmented Generation** adds one step: give the retrieved evidence to an LLM and ask it to write the answer *using only that evidence*, with citations.
The retrieval quality we've just measured sets the ceiling: *an LLM cannot cite a document the retriever never returned* - and it will happily improvise if you let it.

The prompt is an **interface contract** between the search service and the model. A good one:
1. states the role and the **rules** (answer only from sources, cite as `[n]`, say so when the sources don't contain the answer),
2. lists numbered **sources with provenance** (doc id, title, date) so citations can be verified,
3. appends the user question *last*,
4. is **not even called** when the search service abstained - cheapest hallucination guard there is.

In [ ]:
RAG_SYSTEM = """You are Northwind's internal knowledge assistant.
Rules:
1. Answer ONLY using the numbered SOURCES below. Do not use outside knowledge.
2. Cite every factual statement with its source number, like [1] or [2][3].
3. If the sources do not contain the answer, reply exactly: "I could not find this in the knowledge base."
4. If sources conflict, prefer the most recent one and mention the date.
5. Keep the answer under 120 words."""

def build_rag_prompt(question, response, max_sources=3):
    """Returns (system, user) strings. Raises if the search service abstained: don't call the LLM without evidence."""
    if not response.confident:
        raise ValueError("search service abstained - answer 'I don't know' without calling the LLM")
    blocks = []
    for n, h in enumerate(response.hits[:max_sources], 1):
        blocks.append(f"[{n}] (doc {h.doc_id} | {h.department} | {h.date} | {h.sensitivity})\n"
                      f"{h.title}" + (f" > {h.section}" if h.section else "") + f"\n{h.text}")
    user = "SOURCES:\n" + "\n\n".join(blocks) + f"\n\nQUESTION: {question}\nANSWER (with citations):"
    return RAG_SYSTEM, user

question = "How many weeks of paid leave does a primary carer get?"
resp = SEARCH.query(question, k=3, role="employee")
system_prompt, user_prompt = build_rag_prompt(question, resp)
print(system_prompt, "\n", "-" * 100)
print(user_prompt)
print("-" * 100)
print(f"~{int((len(system_prompt) + len(user_prompt)) / 4)} prompt tokens (rule of thumb: 4 chars per token). "
      f"Note: freshness filtering means the 2022 policy (16 weeks) was never offered to the model.")

try:
    build_rag_prompt("what is the dress code for client meetings", SEARCH.query("what is the dress code for client meetings", role="employee"))
except ValueError as e:
    print("\nNo-answer path ->", e)

In [ ]:
# ---- OPTIONAL: plug in your organisation's approved LLM (Day 3 covers the generation side in depth) ----
def call_llm(system_prompt, user_prompt):
    """Connect your approved LLM here (Azure OpenAI, Amazon Bedrock, Vertex AI, ...) and return the answer text."""
    raise NotImplementedError

try:
    print(call_llm(system_prompt, user_prompt))
except NotImplementedError:
    print("skipped: no LLM connected. (The grounded prompt above is the deliverable of this lab; generation is Day 3.)")

## 11. Reference architecture - from this notebook to production

What we built in 150 lines is the *query path*. In production, the interesting engineering is everything around it. The picture below is the whiteboard you'll draw with clients.

In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

def flow(ax, y, labels, colors, title):
    ax.text(0.0, y + 0.62, title, fontsize=11, fontweight="bold", color=INK)
    n = len(labels); w = 0.92 / n
    for i, (lab, col) in enumerate(zip(labels, colors)):
        x = i * (w + 0.0) + 0.0
        ax.add_patch(FancyBboxPatch((x + 0.01, y), w - 0.035, 0.5, boxstyle="round,pad=0.01,rounding_size=0.02", fc=col, ec=INK, lw=0.8))
        ax.text(x + w / 2 - 0.012, y + 0.25, lab, ha="center", va="center", fontsize=8, color="white" if col in (TEAL, INK, CORAL) else INK)
        if i < n - 1:
            ax.add_patch(FancyArrowPatch((x + w - 0.025, y + 0.25), (x + w + 0.012, y + 0.25), arrowstyle="-|>", mutation_scale=9, color=INK))

fig, ax = plt.subplots(figsize=(13, 4.6)); ax.set_xlim(0, 0.97); ax.set_ylim(-0.1, 2.3); ax.axis("off")
flow(ax, 1.5, ["Connectors\nSharePoint, Confluence,\nServiceNow, mail", "Parse\nPDF / OCR / HTML\n+ metadata & ACLs", "Clean & dedupe\nPII scrub,\nversion detection",
               "Chunk\nheading-aware +\ncontext header", "Embed\nmodel@version\n(batch)", "Index\nvector + BM25 +\nmetadata filters"],
     [GOLD, GOLD, GOLD, LIME, TEAL, INK], "INGESTION (offline / event-driven)")
flow(ax, 0.2, ["Query +\nuser identity", "ACL -> filter\n(pre-filter)", "Embed query\n(same model!)", "Hybrid retrieve\nBM25 + dense\n-> RRF", "Rerank\ncross-encoder\n(optional)", "Threshold\n/ abstain", "Results + cites\nor RAG prompt"],
     [GOLD, CORAL, TEAL, TEAL, LIME, CORAL, INK], "QUERY PATH (online, p95 target < 300 ms before the LLM)")
plt.tight_layout(); plt.savefig(ART / "architecture.png"); plt.show()

### Ingestion: where most projects really fail
* **Connectors** with incremental sync and **deletion propagation** (a retired policy must vanish from the index - the #1 enterprise-search trust killer is stale or deleted content still being served).
* **Parsing:** native PDFs vs scanned images (OCR), tables, slides, e-mail threads, images with text. Parsing quality is the hidden ceiling on everything downstream; evaluate it separately.
* **Metadata & permissions captured at ingestion** (owner, dates, sensitivity, ACL group IDs) - retrofitting is painful.
* **Clean:** boilerplate removal, near-duplicate detection (the 2022/2024 problem), PII handling per policy.
* **Idempotent, observable pipeline:** every chunk records `source_doc_version`, `chunker_version`, `embedding_model@version`, `ingested_at`.

### Query path
Authenticate -> derive the user's group IDs -> pre-filter -> (optional query rewriting / expansion) -> hybrid retrieval -> (optional rerank) -> threshold -> response with citations. Cache popular queries; log everything (privately - queries are sensitive too).

### Changing the embedding model = re-index (plan for it on day 1)
Vectors from model A and model B live in *different, incomparable spaces*. Never mix them in one index. Blue/green pattern:
1. build `index_v2` in the background with the new model, alongside the live `index_v1`;
2. run the **golden set** on both - ship only if v2 wins (and no query type regresses);
3. switch traffic (alias swap), keep v1 for a rollback window, then delete.
A guard in code (below) refuses to query an index with a mismatching model - a silent mismatch returns plausible-looking garbage.

In [ ]:
manifest = {"index_version": "kb-2024-10-v1", "embedding_model": EMB.kind, "dim": int(EMB.dim), "chunker": "heading_chunks(max_words=150)+context_header",
            "n_docs": int(len(docs)), "n_chunks": int(len(CH)),
            "corpus_fingerprint": hashlib.sha1("".join(sorted(docs.filename)).encode()).hexdigest()[:12]}
(ART / "index_manifest.json").write_text(json.dumps(manifest, indent=2))

def assert_compatible(manifest_path, embedder):
    m = json.loads(Path(manifest_path).read_text())
    if (m["embedding_model"], m["dim"]) != (embedder.kind, embedder.dim):
        raise RuntimeError(f"Index built with {m['embedding_model']}/{m['dim']} but query embedder is {embedder.kind}/{embedder.dim}: re-index!")
    return m

print(json.dumps(assert_compatible(ART / "index_manifest.json", EMB), indent=2))
class _FakeNewModel:  kind, dim = "some-other-model-v2", 768
try:
    assert_compatible(ART / "index_manifest.json", _FakeNewModel)
except RuntimeError as e:
    print("\nGuard works ->", e)

### Monitoring - what to put on the dashboard
| signal | why it matters | how |
|---|---|---|
| **zero-result / abstain rate** | rising = content gaps or broken ingestion | count responses with `confident = False` |
| **click-through & position of first click** | the real-world MRR | log result impressions and clicks |
| **reformulation rate** | user searched again within 30 s = we failed | session analysis |
| **nDCG@k / Recall@k on the golden set, nightly** | detects silent regressions (model change, chunker change, new connector) | the `evaluate()` function you wrote today, in CI |
| **freshness lag** | time from document edit to searchable | timestamp diff in the pipeline |
| **filtered-recall & ACL tests** | security regressions | canary users with known entitlements |
| **latency p50/p95/p99, cost per 1k queries** | SLOs and budget | tracing, per-stage timings (see `Response.latency_ms`) |
| **thumbs-up/down + "no answer found" feedback** | grows the golden set from real queries | UI widget -> labeling queue |

### Cost and latency - orders of magnitude (relative, no prices)
| component | latency | cost driver | knob |
|---|---|---|---|
| query embedding (small model, CPU) | ~5-30 ms | compute per query | model size, batching, caching |
| query embedding (hosted API) | ~50-300 ms network | per-token fee | batch, cache, regional endpoint |
| BM25 / inverted index | ~1-10 ms | RAM / disk | shards, replicas |
| dense search (ANN) | ~1-20 ms | **RAM** (vectors x dims x 4 bytes), index type | HNSW `ef`, IVF `nprobe`, quantisation (PQ/int8) |
| cross-encoder rerank (top 30) | ~50-400 ms | compute per (query, passage) | rerank depth, distilled models, GPU |
| LLM generation (Day 3) | ~1-10 s | **tokens in + out** | fewer/shorter chunks, caching, smaller model |
| indexing (batch) | minutes-hours | one-off embedding compute + re-embedding on model change | incremental ingestion, versioned indexes |

*Memory sanity check:* 10M chunks x 384 dims x 4 bytes = 15 GB raw (HNSW graph adds more). With int8 quantisation ~4x smaller; with PQ ~10-30x. That arithmetic - not the brand name - often decides the architecture.

## 12. Wrap-up

### Five takeaways
1. **Measure first.** A golden set + `evaluate()` turned every design argument into a number. Keep it in CI; grow it from real queries.
2. **No single retriever wins.** Lexical is best for identifiers, dense for paraphrase; **hybrid with RRF** is the pragmatic default.
3. **Chunking and metadata are architecture, not plumbing.** Boundaries, contextual headers, and provenance decide answer quality, citations, filters and deletes.
4. **Trust is a feature:** pre-filter for access control, handle versions/freshness, and **abstain** when evidence is weak.
5. **Embedding model = versioned dependency.** Plan for re-indexing, store model versions, and test before switching.

### Architect's checklist for any search / vector system
- [ ] Do we have a golden query set with the four types (semantic, keyword, hybrid, no-answer) and an owner?
- [ ] Which metrics and targets? (Recall@k for RAG, nDCG/MRR for human search, abstention precision)
- [ ] Hybrid retrieval, or evidence that dense-only is enough on *our* queries?
- [ ] Chunking strategy chosen by experiment; provenance (doc/chunk id, version, ACL) on every chunk?
- [ ] ACL enforced as a **pre-filter** with the user's *current* entitlements? Filtered recall tested?
- [ ] Versioning/freshness rule and deletion propagation defined?
- [ ] Confidence threshold + "I don't know" UX; what happens to low-confidence queries (log, feedback)?
- [ ] Scale arithmetic done: vectors x dims x bytes, QPS, latency budget, ANN vs exact?
- [ ] Embedding model pinned and versioned; blue/green re-index runbook exists?
- [ ] Multilingual and domain-jargon needs checked; PII and data-residency constraints reviewed?
- [ ] Monitoring: zero-result rate, CTR, nightly golden-set nDCG, freshness lag, p95 latency?

### Homework / stretch tasks
1. **Query expansion:** add 10 new gold queries of your own (include two no-answer). Which type regresses first when you change the chunk size?
2. **Better stemming / typo tolerance:** replace `light_stem` with a proper stemmer or add character n-gram TF-IDF; how much of the BM25 semantic gap does it close? (Spoiler: little.)
3. **Embedding model bake-off:** try another model supported by `fastembed` (`TextEmbedding.list_supported_models()`), re-run the leaderboard, and write the 5-line recommendation you'd give the client (quality, size, latency, languages).
4. **Real reranker:** plug in a cross-encoder from `fastembed` (`TextCrossEncoder`) at depth 20 and measure nDCG@5 vs the lexical stand-in.
5. **HNSW intuition:** implement a greedy graph walk over a k-NN graph of the 100k synthetic vectors and compare to IVF at equal recall.
6. **Break it:** construct a query where hybrid is *worse* than dense alone. What does that tell you about RRF's assumptions?
7. **Security drill:** add a `department`-scoped ACL (HR-only documents) and write a test that proves a Finance user can never retrieve them via any query.

*Next up (Day 3): retrieval-augmented generation, agents and evaluation of generated answers - all standing on the retrieval service you just built.*